# Analyse en Voorspellingen van Wedstrijden in de Jupiler Pro League

**Academisch project / Data-analyse notebook**

Auteur: Torben de Schrijver
Datum: [2026]

Beschrijving: Deze notebook is bedoeld om factoren te analyseren die voetbalwedstrijden beïnvloeden, en om voorspellingen van experts te vergelijken met datagedreven modellen.

## 0. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pyodbc
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

from scipy.stats import ttest_ind, chisquare, pearsonr, chi2_contingency
import re

## 1. Introductie

### Doel van de analyse
Het doel van deze analyse is te onderzoeken hoe expertinschattingen van voetbalwedstrijden overeenkomen met data-gedreven voorspellingen op basis van historische wedstrijdstatistieken.

### Onderzoeksvraag
In welke mate zijn de voorspellingen van experts (journalisten, analisten, trainers) accuraat vergeleken met analyses gemaakt met programmeercode en machine learning modellen?

### Dataset beschrijving
De dataset bevat historische wedstrijddata van de Jupiler Pro League, inclusief:
- Wedstrijdresultaten
- Aantal doelpunten en kaarten
- Teamstatistieken (schoten, corners, ranglijstpositie, thuis/uit)
- Weersomstandigheden en stadioninformatie  
Daarnaast bevat de dataset ingevulde vragenlijsten van experts met hun voorspellingen en factorbeoordelingen.

### Overzicht van de methode
1. **Data verzamelen**: Historische wedstrijden en ingevulde vragenlijsten van experts.  
2. **Feature selectie**: Factoren zoals thuisvoordeel, recente vorm, rangverschil, weersomstandigheden.  
3. **Vergelijking voorspellingen**: Experts vs data-gedreven modellen.  
4. **Analyse**: Accuracy, bias-analyse, invloed van factoren en correlaties tussen expertinschattingen en resultaten.  
5. **Visualisatie en rapportage**: Grafieken, tabellen en samenvatting van inzichten.

## 2. Data laden

- Wedstrijddataset
- Weersdata
- Teams / Ranglijst
- Experts dataset

Commentaar over de structuur van de data

### Database connectie instellen

In [4]:
server = 'localhost'
database = 'JPL_Data'
username = 'tds'
password = 'tds'

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
#    "Trusted_Connection=yes;"
)
conn = pyodbc.connect(conn_str)

### Tabellen inladen

In [5]:
# Team
team_df = pd.read_sql("SELECT * FROM Team;", conn)

# Wedstrijd
wedstrijd_df = pd.read_sql("SELECT * FROM Wedstrijd;", conn)

# WedstrijdStatistiek
statistiek_df = pd.read_sql("SELECT * FROM WedstrijdStatistiek;", conn)

# WedstrijdWeer
weer_df = pd.read_sql("SELECT * FROM WedstrijdWeer;", conn)

# WedstrijdWeerVooraf
weer_vooraf_df = pd.read_sql("SELECT * FROM WedstrijdWeerVooraf;", conn)

## 3. Data cleaning

### Wedstrijddata

In [6]:
wedstrijd_df.info()

wedstrijd_df = wedstrijd_df.drop(['divisie'], axis=1)


<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   wedstrijd_id     2008 non-null   int64 
 1   divisie          2008 non-null   str   
 2   wedstrijd_datum  2008 non-null   object
 3   wedstrijd_tijd   2008 non-null   object
 4   thuis_team       2008 non-null   str   
 5   uit_team         2008 non-null   str   
dtypes: int64(1), object(2), str(3)
memory usage: 94.2+ KB


In [7]:
print(statistiek_df.info())

# 1. Selecteer float64 kolommen
float_cols = statistiek_df.select_dtypes(include=['float64']).columns

# 2. Vul NaN met gemiddelde van elke kolom
statistiek_df[float_cols] = statistiek_df[float_cols].fillna(statistiek_df[float_cols].mean())

# 3. Rond af en converteer naar gewone int64
statistiek_df[float_cols] = statistiek_df[float_cols].round(0).astype('int')

# Check
print(statistiek_df.info())

In [8]:
team_df.info()

team_df = team_df.drop(['latitude', 'longitude'], axis=1)

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   team_naam           24 non-null     str    
 1   latitude            24 non-null     float64
 2   longitude           24 non-null     float64
 3   stadion_capaciteit  24 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 896.0 bytes


In [9]:
weer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   wedstrijd_id                    2008 non-null   int64  
 1   temperatuur_c                   2008 non-null   float64
 2   neerslag_mm                     2008 non-null   float64
 3   relatieve_luchtvochtigheid_pct  2008 non-null   int64  
 4   windsnelheid_m_s                2008 non-null   float64
 5   windrichting_graden             2008 non-null   int64  
 6   windstoten_m_s                  2008 non-null   float64
 7   bewolking_pct                   2008 non-null   int64  
 8   weercode                        2008 non-null   int64  
 9   luchtdruk_hpa                   2008 non-null   float64
 10  temperatuur_gem_c               2008 non-null   float64
 11  neerslag_som_mm                 2008 non-null   float64
 12  windsnelheid_gem_m_s            2008 non-null

In [10]:
weer_vooraf_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   wedstrijd_id                  2008 non-null   int64  
 1   temperatuur_gem_laatste48_c   2008 non-null   float64
 2   neerslag_som_laatste48_mm     2008 non-null   float64
 3   windstoten_max_laatste48_m_s  2008 non-null   float64
 4   regen_uren_laatste48          2008 non-null   int64  
 5   hitte_uren_laatste48          2008 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 94.2 KB


#### Tabellen mergen

In [11]:
wedstrijden_compleet_df = (
    wedstrijd_df
    .merge(statistiek_df, on='wedstrijd_id', how='left')
    .merge(weer_df, on='wedstrijd_id', how='left')
    .merge(weer_vooraf_df, on='wedstrijd_id', how='left')
)

# merge met team info (thuis en uit)
wedstrijden_compleet_df = (
    wedstrijden_compleet_df
    .merge(team_df.add_prefix('thuis_'), left_on='thuis_team', right_on='thuis_team_naam', how='left')
    .merge(team_df.add_prefix('uit_'), left_on='uit_team', right_on='uit_team_naam', how='left')
)

wedstrijden_compleet_df = wedstrijden_compleet_df.drop('wedstrijd_id', axis=1)

print("Shape van complete dataset:", wedstrijden_compleet_df.shape)
wedstrijden_compleet_df.info()
wedstrijden_compleet_df.head()

## 4. Feature engineering

In [12]:
# Kopieer je dataframe voor veiligheid
df = wedstrijden_compleet_df.copy()

### Seizoenkolom toevoegen

In [13]:
def seizoen_bepalen(datum):
    jaar = datum.year
    if datum.month >= 7:  # juli-december → nieuw seizoen startjaar
        return f"{jaar}/{jaar+1}"
    else:                # januari-juni → vorig jaar als startjaar
        return f"{jaar-1}/{jaar}"

# Voeg kolom toe
df['seizoen'] = df['wedstrijd_datum'].apply(seizoen_bepalen)

# Controleer relevante kolommen
df[['wedstrijd_datum', 'wedstrijd_tijd', 'thuis_team', 'uit_team', 'resultaat_voltijd', 'seizoen']].head()

### Rangschikking

In [14]:
# Sorteer op datum per seizoen
df = df.sort_values(['seizoen','wedstrijd_datum'])

# Kolommen voor rang
df['thuis_rank'] = 0
df['uit_rank'] = 0

# Itereer per seizoen
for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen']==seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team','uit_team']].values.ravel())
    
    # Initialiseer dict
    stats_dict = {team: {'punten':0, 'doelpunten_voor':0, 'doelpunten_tegen':0} for team in teams}
    
    ranglijst_thuis = []
    ranglijst_uit = []
    
    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']
        
        # Punten toevoegen
        if resultaat == 'H':
            stats_dict[thuis]['punten'] += 3
        elif resultaat == 'A':
            stats_dict[uit]['punten'] += 3
        elif resultaat == 'D':
            stats_dict[thuis]['punten'] += 1
            stats_dict[uit]['punten'] += 1
        
        # Doelpunten bijwerken
        doel_thuis = row['doelpunten_thuis_voltijd'] if pd.notnull(row['doelpunten_thuis_voltijd']) else 0
        doel_uit = row['doelpunten_uit_voltijd'] if pd.notnull(row['doelpunten_uit_voltijd']) else 0
        
        stats_dict[thuis]['doelpunten_voor'] += doel_thuis
        stats_dict[thuis]['doelpunten_tegen'] += doel_uit
        
        stats_dict[uit]['doelpunten_voor'] += doel_uit
        stats_dict[uit]['doelpunten_tegen'] += doel_thuis
        
        # Bereken doelpuntenverschil
        for team in teams:
            stats_dict[team]['doelpuntenverschil'] = stats_dict[team]['doelpunten_voor'] - stats_dict[team]['doelpunten_tegen']
        
        # Sorteer op punten, dan doelpuntenverschil
        sorted_teams = sorted(stats_dict.items(), key=lambda x: (-x[1]['punten'], -x[1]['doelpuntenverschil'], x[0]))
        team_rank = {team:rank+1 for rank, (team, stats) in enumerate(sorted_teams)}
        
        ranglijst_thuis.append(team_rank[thuis])
        ranglijst_uit.append(team_rank[uit])
    
    df.loc[df['seizoen']==seizoen, 'thuis_rank'] = ranglijst_thuis
    df.loc[df['seizoen']==seizoen, 'uit_rank'] = ranglijst_uit

# Check
df[['wedstrijd_datum','thuis_team','uit_team','resultaat_voltijd','seizoen','thuis_rank','uit_rank']].tail(15)

### Titelstrijd / degradatiestrijd

In [ ]:
# Voeg velden toe
df['titel_match'] = 0
df['degradatie_match'] = 0

# Sorteer op datum per seizoen
df = df.sort_values(['seizoen', 'wedstrijd_datum'])

for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen'] == seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team', 'uit_team']].values.ravel())
    
    # Initialiseer stats
    stats_dict = {team: {'punten':0, 'doelpunten_voor':0, 'doelpunten_tegen':0} for team in teams}
    
    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']
        
        # Punten
        if resultaat == 'H':
            stats_dict[thuis]['punten'] += 3
        elif resultaat == 'A':
            stats_dict[uit]['punten'] += 3
        elif resultaat == 'D':
            stats_dict[thuis]['punten'] += 1
            stats_dict[uit]['punten'] += 1
        
        # Doelpunten
        doel_thuis = row['doelpunten_thuis_voltijd'] if pd.notnull(row['doelpunten_thuis_voltijd']) else 0
        doel_uit = row['doelpunten_uit_voltijd'] if pd.notnull(row['doelpunten_uit_voltijd']) else 0
        stats_dict[thuis]['doelpunten_voor'] += doel_thuis
        stats_dict[thuis]['doelpunten_tegen'] += doel_uit
        stats_dict[uit]['doelpunten_voor'] += doel_uit
        stats_dict[uit]['doelpunten_tegen'] += doel_thuis
        
        # Doelpuntenverschil
        for team in teams:
            stats_dict[team]['doelpuntenverschil'] = stats_dict[team]['doelpunten_voor'] - stats_dict[team]['doelpunten_tegen']
        
        # Ranglijst op punten en doelpuntenverschil
        sorted_teams = sorted(stats_dict.items(), key=lambda x: (-x[1]['punten'], -x[1]['doelpuntenverschil'], x[0]))
        team_rank = {team: rank+1 for rank, (team, stats) in enumerate(sorted_teams)}
        
        thuis_rank = team_rank[thuis]
        uit_rank = team_rank[uit]
        punten_thuis = stats_dict[thuis]['punten']
        punten_uit = stats_dict[uit]['punten']
        
        # Titelwedstrijd: beide in top2, >40 punten, max 5 punten verschil
        if thuis_rank <= 2 and uit_rank <= 2 and punten_thuis > 40 and punten_uit > 40 and abs(punten_thuis - punten_uit) <= 5:
            df.loc[idx, 'titel_match'] = 1
        
        # Degradatiewedstrijd: seizoen >20 wedstrijden, beide in laatste 3, max 3 punten verschil
        if idx > 20:  # indicatief: seizoen gevorderd
            max_rank = len(teams)
            if thuis_rank >= max_rank-2 and uit_rank >= max_rank-2 and abs(punten_thuis - punten_uit) <= 6:
                df.loc[idx, 'degradatie_match'] = 1

In [16]:
# Toon enkele wedstrijden voor titelstrijd
titel_wedstrijden = df[df['titel_match'] == 1]
print("Voorbeeld Titelwedstrijden:")
print(titel_wedstrijden[['wedstrijd_datum', 'wedstrijd_tijd','thuis_team','uit_team','thuis_rank','uit_rank','titel_match']].head(10))

# Toon enkele wedstrijden voor degradatiestrijd
degradatie_wedstrijden = df[df['degradatie_match'] == 1]
print("\nVoorbeeld Degradatiewedstrijden:")
print(degradatie_wedstrijden[['wedstrijd_datum','wedstrijd_tijd','thuis_team','uit_team','thuis_rank','uit_rank','degradatie_match']].head(10))

### Rivalen / Derby

In [17]:
# Lijst van echte Belgische derby's / klassieke rivalen
klassieke_rivalen = [
    ("Antwerp", "Beerschot VA"),            # Antwerpse derby
    ("RSC Anderlecht", "Club Brugge"),      # De Topper / Belgische klassieker
    ("Club Brugge", "Cercle Brugge"),       # Brugse derby
    ("Club Brugge", "Union Saint-Gilloise"),# Moderne rivaliteit
    ("Standard", "RSC Anderlecht")          # Belgian Clasico
]

# Functie om te controleren of een wedstrijd een derby/rivaliteit is
def is_derby(row):
    thuis = row['thuis_team']
    uit = row['uit_team']
    return int(any((thuis, uit) == pair or (uit, thuis) == pair for pair in klassieke_rivalen))

# Voeg het veld toe aan je dataframe
df['derby/rivaal'] = df.apply(is_derby, axis=1)

# Check een paar voorbeelden
print(df[['thuis_team', 'uit_team', 'derby/rivaal']].head(20))

          thuis_team          uit_team  derby/rivaal
0               Genk          Kortrijk             0
1      Cercle Brugge          Standard             0
2         St Truiden          Mouscron             0
3            Waregem          Mechelen             0
4   Waasland-Beveren       Club Brugge             0
5         Anderlecht          Oostende             0
6          Charleroi              Gent             0
7              Eupen           Antwerp             0
8        Club Brugge        St Truiden             0
9           Standard           Waregem             0
10          Kortrijk         Charleroi             0
11          Oostende     Cercle Brugge             0
12          Mechelen              Genk             0
13              Gent             Eupen             0
14          Mouscron        Anderlecht             0
15           Antwerp  Waasland-Beveren             0
16        Anderlecht          Mechelen             0
17          Oostende       Club Brugge        

### Aanvallende of Verdedigende ploeg

In [18]:
import pandas as pd
import numpy as np

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. LONG FORMAT
# ===============================
home = df[[
    "wedstrijd_datum", "thuis_team",
    "schoten_thuis", "doelpunten_thuis_voltijd", "corners_thuis",
    "schoten_uit", "doelpunten_uit_voltijd"
]].copy()

home.columns = [
    "datum", "team",
    "shots_for", "goals_for", "corners_for",
    "shots_against", "goals_against"
]

away = df[[
    "wedstrijd_datum", "uit_team",
    "schoten_uit", "doelpunten_uit_voltijd", "corners_uit",
    "schoten_thuis", "doelpunten_thuis_voltijd"
]].copy()

away.columns = home.columns

matches_long = pd.concat([home, away])

# ===============================
# 3. ROLLING PER TEAM (FIX)
# ===============================
matches_long = matches_long.sort_values(["team", "datum"])

rolling_list = []

for team, group in matches_long.groupby("team"):
    
    group = group.sort_values("datum").copy()
    
    rolled = group[[
        "shots_for", "goals_for", "corners_for",
        "shots_against", "goals_against"
    ]].rolling(window=5, min_periods=3).mean().shift(1)
    
    rolled["team"] = team
    rolled["datum"] = group["datum"].values
    
    rolling_list.append(rolled)

rolling = pd.concat(rolling_list)

# ===============================
# 4. MERGE TERUG
# ===============================
rolling_home = rolling.rename(columns={
    "team": "thuis_team",
    "shots_for": "home_shots_for",
    "goals_for": "home_goals_for",
    "corners_for": "home_corners_for",
    "shots_against": "home_shots_against",
    "goals_against": "home_goals_against",
    "datum": "wedstrijd_datum"
})

rolling_away = rolling.rename(columns={
    "team": "uit_team",
    "shots_for": "away_shots_for",
    "goals_for": "away_goals_for",
    "corners_for": "away_corners_for",
    "shots_against": "away_shots_against",
    "goals_against": "away_goals_against",
    "datum": "wedstrijd_datum"
})

df = df.merge(rolling_home, on=["thuis_team", "wedstrijd_datum"], how="left")
df = df.merge(rolling_away, on=["uit_team", "wedstrijd_datum"], how="left")

# ===============================
# 5. STYLE SCORE
# ===============================
home_attack = df["home_goals_for"] + 0.3 * df["home_shots_for"]
home_defense = df["home_goals_against"] + 0.3 * df["home_shots_against"]

away_attack = df["away_goals_for"] + 0.3 * df["away_shots_for"]
away_defense = df["away_goals_against"] + 0.3 * df["away_shots_against"]

df["home_style_score"] = home_attack - home_defense
df["away_style_score"] = away_attack - away_defense

# ===============================
# 6. CLASSIFICATIE
# ===============================
all_scores = pd.concat([df["home_style_score"], df["away_style_score"]])

low = all_scores.quantile(0.33)
high = all_scores.quantile(0.66)

def classify(x):
    if x >= high:
        return "Aanvallend"
    elif x <= low:
        return "Verdedigend"
    else:
        return "Gebalanceerd"

df["thuis_speelstijl"] = df["home_style_score"].apply(classify)
df["uit_speelstijl"] = df["away_style_score"].apply(classify)

df["speelstijl_combo"] = df["thuis_speelstijl"] + "_vs_" + df["uit_speelstijl"]

# ===============================
# RESULTAAT
# ===============================
print(df[[
    "thuis_team", "uit_team",
    "thuis_speelstijl", "uit_speelstijl",
    "speelstijl_combo"
]].head())

         thuis_team     uit_team thuis_speelstijl uit_speelstijl  \
0              Genk     Kortrijk     Gebalanceerd   Gebalanceerd   
1     Cercle Brugge     Standard     Gebalanceerd   Gebalanceerd   
2        St Truiden     Mouscron     Gebalanceerd   Gebalanceerd   
3           Waregem     Mechelen     Gebalanceerd   Gebalanceerd   
4  Waasland-Beveren  Club Brugge     Gebalanceerd   Gebalanceerd   

               speelstijl_combo  
0  Gebalanceerd_vs_Gebalanceerd  
1  Gebalanceerd_vs_Gebalanceerd  
2  Gebalanceerd_vs_Gebalanceerd  
3  Gebalanceerd_vs_Gebalanceerd  
4  Gebalanceerd_vs_Gebalanceerd  


### Recente Vorm (laatste 5 wedstrijden)

In [19]:
# Kolommen aanmaken
df['thuis_form_5'] = np.nan
df['uit_form_5'] = np.nan

for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen'] == seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team','uit_team']].values.ravel())
    
    form_dict = {team: [] for team in teams}

    thuis_form = []
    uit_form = []

    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']

        # Vorm berekenen (NaN als geen geschiedenis)
        thuis_form.append(np.mean(form_dict[thuis][-5:]) if len(form_dict[thuis]) > 0 else np.nan)
        uit_form.append(np.mean(form_dict[uit][-5:]) if len(form_dict[uit]) > 0 else np.nan)

        # Resultaat toevoegen
        if resultaat == 'H':
            form_dict[thuis].append(3)
            form_dict[uit].append(0)
        elif resultaat == 'A':
            form_dict[thuis].append(0)
            form_dict[uit].append(3)
        else:
            form_dict[thuis].append(1)
            form_dict[uit].append(1)

    df.loc[df['seizoen']==seizoen,'thuis_form_5'] = thuis_form
    df.loc[df['seizoen']==seizoen,'uit_form_5'] = uit_form

### H2H

In [20]:
def calc_h2h_last3(df):
    h2h_thuis = []
    h2h_uit = []

    for idx, row in df.iterrows():

        thuis = row["thuis_team"]
        uit = row["uit_team"]
        datum = row["wedstrijd_datum"]

        # vorige confrontaties tussen dezelfde teams
        h2h_past = df[
            (df["thuis_team"] == thuis) &
            (df["uit_team"] == uit) &
            (df["wedstrijd_datum"] < datum)
        ].sort_values("wedstrijd_datum", ascending=False).head(3)

        # tel wins in laatste 3
        h2h_thuis.append((h2h_past["resultaat_voltijd"] == "H").sum())
        h2h_uit.append((h2h_past["resultaat_voltijd"] == "A").sum())

    return h2h_thuis, h2h_uit


# ===============================
# APPLY
# ===============================
df["h2h_thuis_wins"], df["h2h_uit_wins"] = calc_h2h_last3(df)

# ===============================
# H2H RESULTAAT FEATURE
# ===============================
df["h2h_resultaat"] = df.apply(
    lambda row: "H" if row["h2h_thuis_wins"] > row["h2h_uit_wins"]
    else ("A" if row["h2h_uit_wins"] > row["h2h_thuis_wins"] else "D"),
    axis=1
)

# cleanup
df.drop(["h2h_thuis_wins", "h2h_uit_wins"], axis=1, inplace=True)

print(df.tail())

### Tijdstip

In [21]:
# Zorg dat 'wedstrijd_datum' datetime is
df['wedstrijd_datum'] = pd.to_datetime(df['wedstrijd_datum'], errors='coerce')

# Dag van de week, 0=maandag, 6=zondag
df['weekday'] = df['wedstrijd_datum'].dt.weekday

# Avondmatch indicator (18:00+) indien tijd beschikbaar
if 'wedstrijd_tijd' in df.columns:
    # Zorg dat tijd ook datetime is
    df['avond_match'] = df['wedstrijd_tijd'].apply(lambda t: 1 if pd.notnull(t) and t.hour >= 18 else 0)
else:
    df['avond_match'] = 0

### Extreme weersomstandigheden

In [22]:
# regen > 2 mm = harde regen, temperatuur >23°C = warm
df['regen'] = df['neerslag_som_mm'].apply(lambda x: 1 if pd.notnull(x) and x>0 else 0)
df['harde_regen'] = df['neerslag_mm'].apply(lambda x: 1 if pd.notnull(x) and x>=2 else 0)
df['warm'] = df['temperatuur_gem_c'].apply(lambda x: 1 if pd.notnull(x) and x>=23 else 0)

In [23]:
df[df['regen'] > 0]

### Speelstijl-combinaties

In [24]:
# Combineer speelstijlen (thuis vs uit blijft behouden)
df['speelstijl_combo'] = df['thuis_speelstijl'] + "_vs_" + df['uit_speelstijl']

# Check
df[['thuis_speelstijl','uit_speelstijl','speelstijl_combo']].head()

df['speelstijl_combo'].tail(10)

1998      Verdedigend_vs_Aanvallend
1999      Aanvallend_vs_Verdedigend
2000      Verdedigend_vs_Aanvallend
2001      Aanvallend_vs_Verdedigend
2002     Verdedigend_vs_Verdedigend
2003    Verdedigend_vs_Gebalanceerd
2004       Aanvallend_vs_Aanvallend
2005     Aanvallend_vs_Gebalanceerd
2006     Gebalanceerd_vs_Aanvallend
2007     Gebalanceerd_vs_Aanvallend
Name: speelstijl_combo, dtype: str

### Weekdag en Weekend

In [25]:
# Weekend indicator: 1 als zaterdag (5) of zondag (6), anders 0
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# Controle
df[['wedstrijd_datum', 'weekday', 'is_weekend']].head(10)

### Stadion capaciteit categoriseren

In [26]:
df['capaciteit_cat'] = pd.qcut(df['thuis_stadion_capaciteit'], q=3, labels=['Klein','Middel','Groot'])

print(df[['thuis_stadion_capaciteit','capaciteit_cat']].head(10))

   thuis_stadion_capaciteit capaciteit_cat
0                     23718          Groot
1                     29062          Groot
2                     14600         Middel
3                     12414          Klein
4                      8190          Klein
5                     22500          Groot
6                     15000         Middel
7                      8363          Klein
8                     29062          Groot
9                     27670          Groot


### Rankverschil en vormverschil

In [27]:
# Zorg dat de kolommen numeriek zijn
df['thuis_rank'] = pd.to_numeric(df['thuis_rank'], errors='coerce')
df['uit_rank'] = pd.to_numeric(df['uit_rank'], errors='coerce')
df['thuis_form_5'] = pd.to_numeric(df['thuis_form_5'], errors='coerce')
df['uit_form_5'] = pd.to_numeric(df['uit_form_5'], errors='coerce')

# Rank verschil: positief → thuisteam hoger gerankt
df['rank_diff'] = df['thuis_rank'] - df['uit_rank']

df["abs_rank_diff"] = df["rank_diff"].abs()

# Vorm verschil: positief → thuisteam recent beter
df['form_diff'] = df['thuis_form_5'] - df['uit_form_5']

# Controle: eerste 10 rijen
df[['thuis_team','uit_team','thuis_rank','uit_rank','rank_diff',
    'thuis_form_5','uit_form_5','form_diff']].tail(10)

### Totalen

In [28]:
# Totaal aantal schoten
df['totaal_schoten'] = df['schoten_thuis'] + df['schoten_uit']

# Totaal aantal doelpunten
df['totaal_doelpunten'] = df['doelpunten_thuis_voltijd'] + df['doelpunten_uit_voltijd']

# Totaal aantal overtredingen
df['totaal_overtredingen'] = df['overtredingen_thuis'] + df['overtredingen_uit']

# Totaal aantal kaarten (geel + rood)
df['totaal_kaarten'] = (
    df['gele_kaarten_thuis'] + df['gele_kaarten_uit'] +
    df['rode_kaarten_thuis'] + df['rode_kaarten_uit']
)

# Check
df[['totaal_schoten', 'totaal_doelpunten', 'totaal_overtredingen', 'totaal_kaarten']].head()

### Alle features

In [29]:
# Print alle kolommen van je dataframe
print("Aantal kolommen:", len(df.columns))
print(df.columns.tolist())

### Verwijderen overbodige/gecorreleerde features

In [30]:
df = df.drop(['wedstrijd_tijd', 'thuis_team_naam', 'uit_team_naam', 'temperatuur_c', 'neerslag_mm', 'windsnelheid_m_s',
              'doelpunten_thuis_halftijd', 'doelpunten_uit_halftijd', 'resultaat_halftijd',
'temperatuur_gem_laatste48_c',
    'neerslag_som_laatste48_mm',
    'windstoten_max_laatste48_m_s',
    'regen_uren_laatste48',
    'hitte_uren_laatste48', 'windrichting_graden', 'uit_stadion_capaciteit'], axis=1)

### Splitsen van te voorspellen wedstijden

In [31]:
# zorg dat datum kolom datetime is
df["wedstrijd_datum"] = pd.to_datetime(df["wedstrijd_datum"])

# cutoff datum
cutoff = pd.to_datetime("2026-03-15")

# train dataframe (alles t/m 15 maart 2026)
df_train = df[df["wedstrijd_datum"] <= cutoff].copy()

# predict dataframe (alles na 15 maart 2026)
df_predict = df[df["wedstrijd_datum"] > cutoff].copy()

print("Train shape:", df_train.shape)
print("Predict shape:", df_predict.shape)

# check welke wedstrijden je moet voorspellen
print(df_predict[["wedstrijd_datum", "thuis_team", "uit_team"]])

Train shape: (2006, 66)
Predict shape: (2, 66)
     wedstrijd_datum  thuis_team       uit_team
2006      2026-03-22  Anderlecht  Cercle Brugge
2007      2026-03-22  St Truiden   St. Gilloise


In [32]:
print("Aantal kolommen:", len(df.columns))
print(df.columns.tolist())

Aantal kolommen: 66
['wedstrijd_datum', 'thuis_team', 'uit_team', 'doelpunten_thuis_voltijd', 'doelpunten_uit_voltijd', 'resultaat_voltijd', 'schoten_thuis', 'schoten_uit', 'schoten_op_doel_thuis', 'schoten_op_doel_uit', 'overtredingen_thuis', 'overtredingen_uit', 'corners_thuis', 'corners_uit', 'gele_kaarten_thuis', 'gele_kaarten_uit', 'rode_kaarten_thuis', 'rode_kaarten_uit', 'relatieve_luchtvochtigheid_pct', 'windstoten_m_s', 'bewolking_pct', 'weercode', 'luchtdruk_hpa', 'temperatuur_gem_c', 'neerslag_som_mm', 'windsnelheid_gem_m_s', 'windstoten_max_m_s', 'thuis_stadion_capaciteit', 'seizoen', 'thuis_rank', 'uit_rank', 'titel_match', 'degradatie_match', 'derby/rivaal', 'home_shots_for', 'home_goals_for', 'home_corners_for', 'home_shots_against', 'home_goals_against', 'away_shots_for', 'away_goals_for', 'away_corners_for', 'away_shots_against', 'away_goals_against', 'home_style_score', 'away_style_score', 'thuis_speelstijl', 'uit_speelstijl', 'speelstijl_combo', 'thuis_form_5', 'uit_

In [33]:
df = df_train.copy()
main_features = df.copy()

### Correlatie tussen stadioncapaciteit en ranking

In [91]:
df[["thuis_stadion_capaciteit", "thuis_rank"]].corr()

## 5. Exploratory Data Analysis (EDA)

- Verdelen van thuis- en uitwinst
- Rangverschil versus resultaat
- Vormanalyse
- Weersomstandigheden
- Schoten / corners
- Grafieken en commentaar

### Algemene structuur & sanity checks

In [34]:
print("Shape:", df.shape)

# Datatypes
display(df.dtypes)

# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
display(missing[missing > 0])

Shape: (2006, 66)


wedstrijd_datum             datetime64[s]
thuis_team                            str
uit_team                              str
doelpunten_thuis_voltijd            int64
doelpunten_uit_voltijd              int64
                                ...      
form_diff                         float64
totaal_schoten                      int64
totaal_doelpunten                   int64
totaal_overtredingen                int64
totaal_kaarten                      int64
Length: 66, dtype: object

form_diff             59
uit_form_5            59
thuis_form_5          59
home_corners_for      36
home_shots_for        36
home_goals_for        36
away_goals_against    36
away_goals_for        36
home_shots_against    36
away_style_score      36
away_shots_against    36
home_style_score      36
away_shots_for        36
home_goals_against    36
away_corners_for      36
dtype: int64

### Target analyse

In [35]:
# Resultaat verdeling (H = thuis wint, D = draw, A = uit wint)
plt.figure(figsize=(6,4))
df['resultaat_voltijd'].value_counts().plot(kind='bar')
plt.title("Verdeling van wedstrijdresultaten")
plt.xlabel("Resultaat")
plt.ylabel("Aantal")
plt.show()

# Doelpunten distributie
plt.figure()
sns.histplot(df['doelpunten_thuis_voltijd'], kde=True)
plt.title("Doelpunten thuis")
plt.show()

plt.figure()
sns.histplot(df['doelpunten_uit_voltijd'], kde=True)
plt.title("Doelpunten uit")
plt.show()

### Match statistieken vs resultaat

In [36]:
stats_cols = [
    'schoten_thuis', 'schoten_uit',
    'schoten_op_doel_thuis', 'schoten_op_doel_uit',
    'corners_thuis', 'corners_uit'
]

for col in stats_cols:
    plt.figure()
    sns.boxplot(x=df['resultaat_voltijd'], y=df[col])
    plt.title(f"{col} vs resultaat")
    plt.show()

### Ranking & form

In [37]:
# Rank difference impact
sns.boxplot(x=df['resultaat_voltijd'], y=df['rank_diff'])
plt.title("Rank difference vs resultaat")
plt.show()

# Form difference impact
sns.boxplot(x=df['resultaat_voltijd'], y=df['form_diff'])
plt.title("Form difference vs resultaat")
plt.show()

# Scatter (extra inzicht)
sns.scatterplot(x=df['rank_diff'], y=df['form_diff'], hue=df['resultaat_voltijd'])
plt.title("Rank vs form vs resultaat")
plt.show()

In [38]:
weather_cols = [
    'temperatuur_gem_c', 'neerslag_som_mm',
    'windsnelheid_gem_m_s', 'bewolking_pct'
]

for col in weather_cols:
    plt.figure()
    sns.boxplot(x=df['resultaat_voltijd'], y=df[col])
    plt.title(f"Weer: {col} vs resultaat")
    plt.show()

In [39]:
# Kruistabel met percentages
regen_resultaat = pd.crosstab(df['regen'], df['resultaat_voltijd'], normalize='index')

display(regen_resultaat)

# Plot
regen_resultaat.plot(kind='bar', stacked=True)
plt.title("Regen vs resultaat (%)")
plt.ylabel("Proportie")
plt.show()

sns.boxplot(x=df['harde_regen'], y=df['doelpunten_thuis_voltijd'])
plt.title("Harde regen vs doelpunten")
plt.show()

df.groupby('regen')['doelpunten_thuis_voltijd'].mean().plot(kind='bar')
plt.title("Gemiddelde goals bij regen vs geen regen")
plt.ylabel("Goals")
plt.show()

### Tijd & kalender

In [40]:
# Weekend effect
sns.countplot(x=df['is_weekend'], hue=df['resultaat_voltijd'])
plt.title("Weekend vs resultaat")
plt.show()

# Avondmatch
sns.boxplot(x=df['avond_match'], y=df['doelpunten_thuis_voltijd'])
plt.title("Avondmatch vs doelpunten")
plt.show()

# Weekday
plt.figure(figsize=(10,4))
sns.countplot(x=df['weekday'], hue=df['resultaat_voltijd'])
plt.title("Dag van week vs resultaat")
plt.show()

In [41]:
plt.figure(figsize=(14,10))
corr = df.corr(numeric_only=True)

sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlatiematrix")
plt.show()

### Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?

In [42]:
# Totaal aantal wedstrijden per resultaat
resultaten = df['resultaat_voltijd'].value_counts()
print("Aantal wedstrijden per resultaat (H/A/D):")
print(resultaten)

# Aantal thuis-, uitoverwinningen en gelijke spelen
thuis_overwinningen = (df['resultaat_voltijd'] == 'H').sum()
uit_overwinningen = (df['resultaat_voltijd'] == 'A').sum()
gelijk = (df['resultaat_voltijd'] == 'D').sum()

print(f"\nThuisoverwinningen: {thuis_overwinningen}")
print(f"Uitoverwinningen: {uit_overwinningen}")
print(f"Gelijke spelen: {gelijk}")

# Percentages berekenen
totaal = len(df)
print("\nPercentage resultaten:")
print(f"Thuiswinst: {thuis_overwinningen/totaal*100:.2f}%")
print(f"Uitwinst: {uit_overwinningen/totaal*100:.2f}%")
print(f"Gelijk: {gelijk/totaal*100:.2f}%")

# Functie voor chi-kwadraat test
def chi_kwadraat_test(observed):
    totaal = sum(observed)
    expected = [totaal / len(observed)] * len(observed)
    chi2_stat, p_value = chisquare(observed, expected)
    return chi2_stat, p_value

chi2_stat, p_value = chi_kwadraat_test([thuis_overwinningen, uit_overwinningen, gelijk])
print("\nChi-kwadraat test voor thuisvoordeel:")
print(f"Chi²-statistiek: {chi2_stat:.2f}")
print(f"P-waarde: {p_value:.4e}")

# Conclusie op basis van p-waarde
if p_value < 0.05:
    print("✅ Thuisvoordeel heeft een SIGNIFICANTE invloed.")
else:
    print("❌ Geen significant thuisvoordeel gevonden.")

# Visualisatie
plt.figure(figsize=(8,6))
sns.countplot(x='resultaat_voltijd', data=df, palette='Set2')
plt.title("Verdeling van resultaten (H / A / D)")
plt.xlabel("Resultaat")
plt.ylabel("Aantal wedstrijden")
plt.show()

Aantal wedstrijden per resultaat (H/A/D):
resultaat_voltijd
H    869
A    648
D    489
Name: count, dtype: int64

Thuisoverwinningen: 869
Uitoverwinningen: 648
Gelijke spelen: 489

Percentage resultaten:
Thuiswinst: 43.32%
Uitwinst: 32.30%
Gelijk: 24.38%

Chi-kwadraat test voor thuisvoordeel:
Chi²-statistiek: 108.93
P-waarde: 2.2143e-24
✅ Thuisvoordeel heeft een SIGNIFICANTE invloed.


In [43]:
# Percentages berekenen
total = 869 + 648 + 489
percentages = {
    'Thuiswinst (H)': 869/total*100,
    'Uitwinst (A)': 648/total*100,
    'Gelijk (D)': 489/total*100
}

# Barplot
plt.figure(figsize=(6,4))
sns.barplot(x=list(percentages.keys()), y=list(percentages.values()), palette="pastel")
plt.ylabel("Percentage (%)")
plt.title("Verdeling van wedstrijduitslagen - Thuisvoordeel")
plt.ylim(0, 50)
for i, v in enumerate(percentages.values()):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center')
plt.show()

### In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  

In [44]:
# Gemiddeld totaal aantal schoten per stadioncapaciteit-categorie
gemiddelde_per_cat = df.groupby('capaciteit_cat')['totaal_schoten'].mean()
print("Gemiddeld totaal aantal schoten per stadioncapaciteit-categorie:")
print(gemiddelde_per_cat)

# Boxplot per categorie
plt.figure(figsize=(8,6))
sns.boxplot(x='capaciteit_cat', y='totaal_schoten', data=df, palette='Set3')
plt.title("Totaal aantal schoten per stadioncapaciteit-categorie")
plt.xlabel("Stadioncapaciteit")
plt.ylabel("Totaal aantal schoten")
plt.show()

# Scatterplot en correlatie (numeriek)
plt.figure(figsize=(8,6))
sns.scatterplot(x='thuis_stadion_capaciteit', y='totaal_schoten', data=df)
plt.title("Totaal aantal schoten vs stadioncapaciteit")
plt.xlabel("Stadioncapaciteit")
plt.ylabel("Totaal aantal schoten")
plt.show()

# Pearson correlatie
corr, p_value = pearsonr(df['thuis_stadion_capaciteit'], df['totaal_schoten'])
print(f"Pearson correlatie: {corr:.3f}")
print(f"P-waarde: {p_value:.4e}")

# Conclusie op basis van p-waarde
if p_value < 0.05:
    print("✅ Er is een statistisch significant verband tussen stadioncapaciteit en totaal aantal schoten.")
else:
    print("❌ Geen significant verband gevonden tussen stadioncapaciteit en totaal aantal schoten.")

Gemiddeld totaal aantal schoten per stadioncapaciteit-categorie:
capaciteit_cat
Klein     24.701513
Middel    24.663730
Groot     25.061873
Name: totaal_schoten, dtype: float64


Pearson correlatie: 0.031
P-waarde: 1.7032e-01
❌ Geen significant verband gevonden tussen stadioncapaciteit en totaal aantal schoten.


### In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  

In [45]:
# Scatterplot: overtredingen uitteam vs stadioncapaciteit
plt.figure(figsize=(8,6))
sns.scatterplot(x='thuis_stadion_capaciteit', y='overtredingen_uit', data=df)
plt.title("Aantal overtredingen uitteam vs stadioncapaciteit")
plt.xlabel("Stadioncapaciteit (thuisteam)")
plt.ylabel("Aantal overtredingen uitteam")
plt.show()

# Gemiddeld aantal overtredingen per categorie stadioncapaciteit
gemiddelde_per_cat = df.groupby('capaciteit_cat')['overtredingen_uit'].mean()
print("Gemiddeld aantal overtredingen uitteam per stadioncapaciteit-categorie:")
print(gemiddelde_per_cat)

# Boxplot per categorie
plt.figure(figsize=(8,6))
sns.boxplot(x='capaciteit_cat', y='overtredingen_uit', data=df, palette='Set3')
plt.title("Aantal overtredingen uitteam per stadioncapaciteit-categorie")
plt.xlabel("Stadioncapaciteit")
plt.ylabel("Aantal overtredingen uitteam")
plt.show()

# Pearson correlatie
corr, p_value = pearsonr(df['thuis_stadion_capaciteit'], df['overtredingen_uit'])
print(f"Pearson correlatie: {corr:.3f}")
print(f"P-waarde: {p_value:.4e}")

# Conclusie op basis van p-waarde
if p_value < 0.05:
    print("✅ Er is een statistisch significant verband tussen stadioncapaciteit en overtredingen van het uitteam.")
else:
    print("❌ Geen significant verband gevonden tussen stadioncapaciteit en overtredingen van het uitteam.")

Gemiddeld aantal overtredingen uitteam per stadioncapaciteit-categorie:
capaciteit_cat
Klein     12.200825
Middel    11.687225
Groot     12.984950
Name: overtredingen_uit, dtype: float64


Pearson correlatie: 0.079
P-waarde: 3.6602e-04
✅ Er is een statistisch significant verband tussen stadioncapaciteit en overtredingen van het uitteam.


### Heeft regen invloed op het aantal doelpunten?

In [46]:
# Subsets: met regen vs zonder regen
met_regen = df[df['regen'] == 1]['totaal_doelpunten']
zonder_regen = df[df['regen'] == 0]['totaal_doelpunten']

# Gemiddelde doelpunten
gemiddelde_met_regen = met_regen.mean()
gemiddelde_zonder_regen = zonder_regen.mean()

print(f"Gemiddeld aantal doelpunten bij regen: {gemiddelde_met_regen:.2f}")
print(f"Gemiddeld aantal doelpunten zonder regen: {gemiddelde_zonder_regen:.2f}")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(x=df['regen'].map({0:"Geen regen", 1:"Regen"}), y=df['totaal_doelpunten'], palette="Set2")
plt.title("Invloed van regen op totaal aantal doelpunten")
plt.xlabel("Weersituatie")
plt.ylabel("Totaal aantal doelpunten")
plt.show()

# Statistische toets (t-test)
t_stat, p_value = ttest_ind(met_regen.dropna(), zonder_regen.dropna(), equal_var=False)
print(f"T-test p-waarde: {p_value:.4e}")

# Conclusie op basis van p-waarde
if p_value < 0.05:
    print("✅ Regen heeft een statistisch significante invloed op het aantal doelpunten.")
else:
    print("❌ Geen significant effect van regen op het aantal doelpunten.")

Gemiddeld aantal doelpunten bij regen: 2.94
Gemiddeld aantal doelpunten zonder regen: 2.83


T-test p-waarde: 1.7900e-01
❌ Geen significant effect van regen op het aantal doelpunten.


### In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt?

In [47]:
# Subsets: wedstrijden met regen vs zonder regen
kaarten_met_regen = df[df['regen'] == 1]['totaal_kaarten']
kaarten_zonder_regen = df[df['regen'] == 0]['totaal_kaarten']

# Gemiddelden
gem_met_regen = kaarten_met_regen.mean()
gem_zonder_regen = kaarten_zonder_regen.mean()

print(f"Gemiddeld aantal kaarten bij regen: {gem_met_regen:.2f}")
print(f"Gemiddeld aantal kaarten zonder regen: {gem_zonder_regen:.2f}")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(x=df['regen'].map({0:"Geen regen", 1:"Regen"}), y=df['totaal_kaarten'], palette="Set3")
plt.title("Invloed van regen op totaal aantal kaarten")
plt.xlabel("Weersituatie")
plt.ylabel("Totaal aantal kaarten")
plt.show()

# Statistische toets (t-test)
t_stat, p_value = ttest_ind(kaarten_met_regen.dropna(), kaarten_zonder_regen.dropna(), equal_var=False)
print(f"T-test p-waarde: {p_value:.4e}")

# Conclusie op basis van p-waarde
if p_value < 0.05:
    print("✅ Regen heeft een statistisch significante invloed op het aantal kaarten.")
else:
    print("❌ Geen significant effect van regen op het aantal kaarten.")

Gemiddeld aantal kaarten bij regen: 4.23
Gemiddeld aantal kaarten zonder regen: 4.10


T-test p-waarde: 2.6602e-01
❌ Geen significant effect van regen op het aantal kaarten.


### Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat?

In [48]:
# Categorisatie (NaN blijft NaN)
def categorize_form(x):
    if pd.isna(x):
        return np.nan
    elif x < 1:
        return 'Slecht'
    elif x < 1.7:
        return 'Normaal'
    else:
        return 'Goed'

df['thuis_form_cat'] = df['thuis_form_5'].apply(categorize_form)
df['uit_form_cat'] = df['uit_form_5'].apply(categorize_form)

# Drop rijen zonder vorm (begin seizoen)
df_analysis = df.dropna(subset=['thuis_form_cat','uit_form_cat'])

# Thuis wint kolom
df_analysis['thuis_wint'] = (df_analysis['resultaat_voltijd'] == 'H').astype(int)

# Crosstab (kans op thuiswinst)
crosstab = pd.crosstab(
    df_analysis['thuis_form_cat'],
    df_analysis['uit_form_cat'],
    values=df_analysis['thuis_wint'],
    aggfunc='mean'
).round(2)

print("Kans op thuiswinst per vormcategorie:")
print(crosstab)

# Visualisatie
plt.figure(figsize=(8,6))
sns.heatmap(crosstab, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title("Kans op thuiswinst op basis van recente vorm")
plt.xlabel("Uitteam Vorm")
plt.ylabel("Thuisteam Vorm")
plt.show()

# Chi-square test
count_table = pd.crosstab(
    [df_analysis['thuis_form_cat'], df_analysis['uit_form_cat']],
    df_analysis['thuis_wint']
)

chi2_stat, p_val, dof, expected = chi2_contingency(count_table)

print(f"\nChi²-statistiek: {chi2_stat:.2f}")
print(f"P-waarde: {p_val:.4e}")

# Conclusie op basis van p-waarde
if p_val < 0.05:
    print("✅ Recente vorm is statistisch SIGNIFICANT bepalend voor thuiswinst.")
else:
    print("❌ Geen significant effect van recente vorm op thuiswinst.")

Kans op thuiswinst per vormcategorie:
uit_form_cat    Goed  Normaal  Slecht
thuis_form_cat                       
Goed            0.47     0.53    0.57
Normaal         0.28     0.47    0.51
Slecht          0.31     0.38    0.41



Chi²-statistiek: 67.31
P-waarde: 1.6797e-11
✅ Recente vorm is statistisch SIGNIFICANT bepalend voor thuiswinst.


### Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd?

In [49]:
# Categoriseer resultaten: 1 = thuis wint, 0 = anders
df['thuis_wint'] = df['resultaat_voltijd'].apply(lambda x: 1 if x=='H' else 0)

# Bin rank_diff volgens nieuwe grenzen
bins = [-100, -10, -5, -2, 0, 2, 5, 10, 100]  # extra uitersten om alles te vangen
labels = ['<-10','-10--5','-5--2','-2-0','0-2','2-5','5-10','>10']
df['rank_diff_cat'] = pd.cut(df['rank_diff'], bins=bins, labels=labels)

# Gemiddelde kans op winst per rank_diff categorie
winst_per_cat = df.groupby('rank_diff_cat')['thuis_wint'].mean().reset_index()
print("Gemiddelde kans op winst van thuisteam per rankverschil categorie:")
print(winst_per_cat)

# Visualisatie: lijnplot
plt.figure(figsize=(10,6))
sns.lineplot(data=winst_per_cat, x='rank_diff_cat', y='thuis_wint', marker='o')
plt.title("Kans dat het thuisteam wint versus rankverschil")
plt.xlabel("Rankverschil (thuis - uit)")
plt.ylabel("Gemiddelde kans op winst (thuisteam)")
plt.grid(True)
plt.show()

# Chi-square test voor onafhankelijkheid
crosstab = pd.crosstab(df['rank_diff_cat'], df['thuis_wint'])
chi2_stat, p_val, dof, expected = chi2_contingency(crosstab)

print(f"\nChi-square test voor invloed van rankverschil op thuisresultaat:")
print(f"Chi²-statistiek: {chi2_stat:.2f}, p-waarde: {p_val:.4e}")

# Conclusie op basis van p-waarde
if p_val < 0.05:
    print("✅ Conclusie: Rangverschil heeft een SIGNIFICANTE invloed op het wedstrijdresultaat.")
else:
    print("❌ Conclusie: Rangverschil lijkt GEEN significant effect te hebben op het resultaat.")

Gemiddelde kans op winst van thuisteam per rankverschil categorie:
  rank_diff_cat  thuis_wint
0          <-10    0.842105
1        -10--5    0.659026
2         -5--2    0.536977
3          -2-0    0.447761
4           0-2    0.328947
5           2-5    0.229839
6          5-10    0.198697
7           >10    0.060440



Chi-square test voor invloed van rankverschil op thuisresultaat:
Chi²-statistiek: 478.06, p-waarde: 4.1741e-99
✅ Conclusie: Rangverschil heeft een SIGNIFICANTE invloed op het wedstrijdresultaat.


### In welke mate hebben eerdere confrontaties invloed in uw voorspelling?

In [50]:
# Maak een kolom die checkt of H2H-favoriet gelijk is aan de echte winnaar
df['h2h_correct'] = df['h2h_resultaat'] == df['resultaat_voltijd']

# Percentage van wedstrijden waarbij H2H correct voorspelde
percentage_correct = df['h2h_correct'].mean() * 100
print(f"Percentage H2H correct: {percentage_correct:.2f}%")

# Crosstab: H2H-resultaat vs echte resultaat (percentages)
tabel_pct = pd.crosstab(df['h2h_resultaat'], df['resultaat_voltijd'], normalize='index')
print("\nCrosstab (per H2H-favoriet, aandeel echte resultaten):")
print(tabel_pct)

# Crosstab: absoluut aantal keren
tabel_count = pd.crosstab(df['h2h_resultaat'], df['resultaat_voltijd'])
print("\nCrosstab (aantal wedstrijden per combinatie):")
print(tabel_count)

# Visualisatie: heatmap percentage
plt.figure(figsize=(8,6))
sns.heatmap(tabel_pct, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("H2H-favoriet vs Werkelijk resultaat (percentage per rij)")
plt.ylabel("H2H-favoriet")
plt.xlabel("Werkelijk resultaat")
plt.show()

# Extra: Hoe vaak wint H2H-favoriet?
h2h_wins_count = df['h2h_correct'].sum()
total_matches = df.shape[0]
print(f"\nAantal wedstrijden gewonnen door H2H-favoriet: {h2h_wins_count} van {total_matches} ({h2h_wins_count/total_matches*100:.2f}%)")

Percentage H2H correct: 38.29%

Crosstab (per H2H-favoriet, aandeel echte resultaten):
resultaat_voltijd         A         D         H
h2h_resultaat                                  
A                  0.405920  0.274841  0.319239
D                  0.335766  0.245742  0.418491
H                  0.253165  0.220816  0.526020

Crosstab (aantal wedstrijden per combinatie):
resultaat_voltijd    A    D    H
h2h_resultaat                   
A                  192  130  151
D                  276  202  344
H                  180  157  374



Aantal wedstrijden gewonnen door H2H-favoriet: 768 van 2006 (38.29%)


### In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd?

In [51]:
# Totaal aantal schoten en corners per wedstrijd
df['totaal_schoten'] = df['schoten_thuis'] + df['schoten_uit']
df['totaal_corners'] = df['corners_thuis'] + df['corners_uit']

# Correlatie berekenen (Pearson)
correlatie, p_value = pearsonr(df['totaal_schoten'], df['totaal_corners'])
print(f"Pearson correlatie tussen totaal schoten en totaal corners: {correlatie:.2f}")
print(f"P-waarde: {p_value:.4e}")

# Interpretatie
if abs(correlatie) > 0.5:
    interpretatie = "sterke"
elif abs(correlatie) > 0.3:
    interpretatie = "matige"
else:
    interpretatie = "zwakke"

print(f"👉 Er is een {interpretatie} positieve samenhang tussen aantal schoten en corners.")

# Visualisatie: scatterplot
plt.figure(figsize=(8,6))
sns.scatterplot(x='totaal_schoten', y='totaal_corners', data=df, alpha=0.6)
sns.regplot(x='totaal_schoten', y='totaal_corners', data=df, scatter=False, color='red')
plt.title("Samenhang tussen totaal aantal schoten en corners")
plt.xlabel("Totaal aantal schoten")
plt.ylabel("Totaal aantal corners")
plt.show()

Pearson correlatie tussen totaal schoten en totaal corners: 0.29
P-waarde: 6.6471e-40
👉 Er is een zwakke positieve samenhang tussen aantal schoten en corners.


### Stadioncapaciteit → overtredingen uitteam

In [52]:
# Scatterplot: overtredingen uitteam vs stadioncapaciteit
sns.scatterplot(x=df['thuis_stadion_capaciteit'], y=df['overtredingen_uit'])
plt.title("Capaciteit vs overtredingen uitteam")
plt.xlabel("Thuis stadioncapaciteit")
plt.ylabel("Aantal overtredingen uitteam")
plt.show()

# Correlatie (numeriek)
corr_numeriek = df['thuis_stadion_capaciteit'].corr(df['overtredingen_uit'])
print(f"Pearson correlatie (numeriek): {corr_numeriek:.3f}")

# Gemiddeld aantal overtredingen per capaciteit categorie
gem_overtredingen_per_cat = df.groupby('capaciteit_cat')['overtredingen_uit'].mean()
print("\nGemiddeld aantal overtredingen uitteam per capaciteit-categorie:")
print(gem_overtredingen_per_cat)

# Visualisatie: boxplot per capaciteit categorie
plt.figure(figsize=(8,6))
sns.boxplot(x='capaciteit_cat', y='overtredingen_uit', data=df, palette='Set3')
plt.title("Aantal overtredingen uitteam per stadioncapaciteit-categorie")
plt.xlabel("Stadioncapaciteit categorie")
plt.ylabel("Aantal overtredingen uitteam")
plt.show()

Pearson correlatie (numeriek): 0.079

Gemiddeld aantal overtredingen uitteam per capaciteit-categorie:
capaciteit_cat
Klein     12.200825
Middel    11.687225
Groot     12.984950
Name: overtredingen_uit, dtype: float64


### Als het hard regent, zullen er meer/evenveel/minder kaarten verwacht worden.

In [53]:
# ===============================
# 1. Subset harde regen
# ===============================
regen_hard_df = df[df["harde_regen"] == 1].copy()

# ===============================
# 2. Gemiddelden berekenen
# ===============================
gemiddelde_regen = regen_hard_df["totaal_kaarten"].mean()
gemiddelde_algemeen = df["totaal_kaarten"].mean()

print(f"Gemiddeld aantal kaarten bij harde regen: {gemiddelde_regen:.2f}")
print(f"Algemeen gemiddeld aantal kaarten: {gemiddelde_algemeen:.2f}")

# ===============================
# 3. Interpretatie
# ===============================
if gemiddelde_regen > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_regen < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij harde regen vallen gemiddeld {conclusie} kaarten per wedstrijd.")

# ===============================
# 4. Visualisatie
# ===============================
plt.figure(figsize=(8,6))

sns.boxplot(
    x=df["harde_regen"].map({0: "Geen harde regen", 1: "Harde regen"}),
    y=df["totaal_kaarten"]
)

plt.title("Aantal kaarten: Harde regen vs Geen harde regen")
plt.xlabel("Weersomstandigheden")
plt.ylabel("Totaal aantal kaarten")
plt.show()

# ===============================
# 5. Statistische test (t-test)
# ===============================
groep_regen = df[df["harde_regen"] == 1]["totaal_kaarten"].dropna()
groep_normaal = df[df["harde_regen"] == 0]["totaal_kaarten"].dropna()

t_stat, p_val = ttest_ind(groep_regen, groep_normaal, equal_var=False)

print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Harde regen heeft een SIGNIFICANT effect op het aantal kaarten.")
else:
    print("❌ Conclusie: Harde regen heeft GEEN significant effect op het aantal kaarten.")

Gemiddeld aantal kaarten bij harde regen: 4.38
Algemeen gemiddeld aantal kaarten: 4.13
👉 Bij harde regen vallen gemiddeld meer kaarten per wedstrijd.



p-waarde: 0.60878
❌ Conclusie: Harde regen heeft GEEN significant effect op het aantal kaarten.


### Als het hard regent, zullen er meer/evenveel/minder doelpunten verwacht worden.

In [54]:
# ===============================
# 1. Subset harde regen
# ===============================
regen_hard_df = df[df["harde_regen"] == 1].copy()

# ===============================
# 2. Gemiddelden berekenen
# ===============================
gemiddelde_regen = regen_hard_df["totaal_doelpunten"].mean()
gemiddelde_algemeen = df["totaal_doelpunten"].mean()

print(f"Gemiddeld aantal doelpunten bij harde regen: {gemiddelde_regen:.2f}")
print(f"Algemeen gemiddeld aantal doelpunten: {gemiddelde_algemeen:.2f}")

# ===============================
# 3. Interpretatie
# ===============================
if gemiddelde_regen > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_regen < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij harde regen vallen gemiddeld {conclusie} doelpunten per wedstrijd.")

# ===============================
# 4. Visualisatie
# ===============================
plt.figure(figsize=(8,6))

sns.boxplot(
    x=df["harde_regen"].map({0: "Geen harde regen", 1: "Harde regen"}),
    y=df["totaal_doelpunten"]
)

plt.title("Aantal doelpunten: Harde regen vs Geen harde regen")
plt.xlabel("Weersomstandigheden")
plt.ylabel("Totaal aantal doelpunten")
plt.show()

# ===============================
# 5. Statistische test (t-test)
# ===============================
groep_regen = df[df["harde_regen"] == 1]["totaal_doelpunten"].dropna()
groep_normaal = df[df["harde_regen"] == 0]["totaal_doelpunten"].dropna()

t_stat, p_val = ttest_ind(groep_regen, groep_normaal, equal_var=False)

print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Harde regen heeft een SIGNIFICANT effect op het aantal doelpunten.")
else:
    print("❌ Conclusie: Harde regen heeft GEEN significant effect op het aantal doelpunten.")

Gemiddeld aantal doelpunten bij harde regen: 3.04
Algemeen gemiddeld aantal doelpunten: 2.86
👉 Bij harde regen vallen gemiddeld meer doelpunten per wedstrijd.



p-waarde: 0.60765
❌ Conclusie: Harde regen heeft GEEN significant effect op het aantal doelpunten.


### Als het thuisteam een aanvallende speelstijl hanteert en het uitteam defensief speelt zullen er gemiddeld meer/zelfde/minder doelpunten vallen.

In [55]:
# Specifieke situatie: Aanvallend vs Verdedigend
subset = df[df['speelstijl_combo'] == 'Aanvallend_vs_Verdedigend']

# Gemiddelden
gem_specifiek = subset['totaal_doelpunten'].mean()
gem_algemeen = df['totaal_doelpunten'].mean()

print(f"Gemiddeld aantal doelpunten (Aanvallend vs Verdedigend): {gem_specifiek:.2f}")
print(f"Algemeen gemiddelde aantal doelpunten: {gem_algemeen:.2f}")

# Interpretatie
if gem_specifiek > gem_algemeen:
    conclusie = "meer"
elif gem_specifiek < gem_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 In deze situatie vallen gemiddeld {conclusie} doelpunten.")

# Overzicht alle combinaties
overzicht = df.groupby('speelstijl_combo')['totaal_doelpunten'].mean().sort_values()
print("\nGemiddeld aantal doelpunten per speelstijl-combinatie:")
print(overzicht)

# Visualisatie: barplot
plt.figure(figsize=(10,6))
overzicht.plot(kind='bar')
plt.title("Gemiddeld aantal doelpunten per speelstijl-combinatie")
plt.xlabel("Speelstijl combinatie")
plt.ylabel("Gemiddeld aantal doelpunten")
plt.xticks(rotation=45)
plt.show()

# Statistische test: t-test (specifieke situatie vs algemeen)
groep1 = subset['totaal_doelpunten'].dropna()
groep2 = df['totaal_doelpunten'].dropna()

t_stat, p_val = ttest_ind(groep1, groep2, equal_var=False)
print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Aanvallend vs Verdedigend situatie heeft een SIGNIFICANT effect op het aantal doelpunten.")
else:
    print("❌ Conclusie: Aanvallend vs Verdedigend situatie lijkt GEEN significant effect te hebben op het aantal doelpunten.")

Gemiddeld aantal doelpunten (Aanvallend vs Verdedigend): 3.19
Algemeen gemiddelde aantal doelpunten: 2.86
👉 In deze situatie vallen gemiddeld meer doelpunten.

Gemiddeld aantal doelpunten per speelstijl-combinatie:
speelstijl_combo
Verdedigend_vs_Verdedigend      2.477477
Verdedigend_vs_Aanvallend       2.779221
Aanvallend_vs_Aanvallend        2.801762
Verdedigend_vs_Gebalanceerd     2.820628
Aanvallend_vs_Gebalanceerd      2.882883
Gebalanceerd_vs_Aanvallend      2.898785
Gebalanceerd_vs_Gebalanceerd    2.930435
Gebalanceerd_vs_Verdedigend     3.000000
Aanvallend_vs_Verdedigend       3.189474
Name: totaal_doelpunten, dtype: float64



p-waarde: 0.01393
✅ Conclusie: Aanvallend vs Verdedigend situatie heeft een SIGNIFICANT effect op het aantal doelpunten.


### Als het erg warm (>23°C) is, zal het aantal schoten meer/zelfde/minder zijn dan gemiddeld.

In [56]:
# Analyse: Warm weer vs Aantal schoten

# Totale schoten per wedstrijd
df['totaal_schoten'] = df['schoten_thuis'] + df['schoten_uit']

# Subset voor warm weer (>23°C)
subset_warm = df[df['temperatuur_gem_c'] > 23].copy()

# Gemiddelden berekenen
gemiddelde_warm = subset_warm['totaal_schoten'].mean()
gemiddelde_algemeen = df['totaal_schoten'].mean()

print(f"Gemiddeld aantal schoten bij warm weer (>23°C): {gemiddelde_warm:.2f}")
print(f"Algemeen gemiddeld aantal schoten: {gemiddelde_algemeen:.2f}")

# Interpretatie (meer/minder/hetzelfde)
if gemiddelde_warm > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_warm < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij warm weer vallen gemiddeld {conclusie} schoten per wedstrijd.")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(
    x=pd.cut(df['temperatuur_gem_c'], bins=[-10,23,50], labels=['≤23°C', '>23°C']),
    y=df['totaal_schoten']
)
plt.title("Aantal schoten per wedstrijd vs Temperatuur")
plt.xlabel("Temperatuur")
plt.ylabel("Totaal aantal schoten")
plt.show()

# Statistische test (t-test)
groep_warm = subset_warm['totaal_schoten'].dropna()
groep_algemeen = df['totaal_schoten'].dropna()

t_stat, p_val = ttest_ind(groep_warm, groep_algemeen, equal_var=False)
print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Warm weer heeft een SIGNIFICANT effect op het aantal schoten per wedstrijd.")
else:
    print("❌ Conclusie: Warm weer lijkt GEEN significant effect te hebben op het aantal schoten per wedstrijd.")

Gemiddeld aantal schoten bij warm weer (>23°C): 24.70
Algemeen gemiddeld aantal schoten: 24.80
👉 Bij warm weer vallen gemiddeld minder schoten per wedstrijd.



p-waarde: 0.87160
❌ Conclusie: Warm weer lijkt GEEN significant effect te hebben op het aantal schoten per wedstrijd.


### Als er een derby is, of rivalen spelen, zal het aantal schoten van beide teams meer/zelfde/minder zijn dan gemiddeld.

In [57]:
# Subset van derby- of rivaliteitswedstrijden
derby_subset = df[df['derby/rivaal'] == 1].copy()

# Gemiddelden berekenen
gemiddelde_derby = derby_subset['totaal_schoten'].mean()
gemiddelde_algemeen = df['totaal_schoten'].mean()

print(f"Gemiddeld aantal schoten bij derby/rivaliteit: {gemiddelde_derby:.2f}")
print(f"Algemeen gemiddeld aantal schoten: {gemiddelde_algemeen:.2f}")

# Interpretatie (meer/minder/hetzelfde)
if gemiddelde_derby > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_derby < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij derby/rivaliteitswedstrijden vallen gemiddeld {conclusie} schoten per wedstrijd.")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(
    x=df['derby/rivaal'].map({0:'Geen derby', 1:'Derby/Rivaal'}),
    y=df['totaal_schoten']
)
plt.title("Aantal schoten per wedstrijd: Derby vs Geen Derby")
plt.xlabel("Wedstrijdtype")
plt.ylabel("Totaal aantal schoten")
plt.show()

# Statistische test (t-test)
groep_derby = derby_subset['totaal_schoten'].dropna()
groep_algemeen = df['totaal_schoten'].dropna()

t_stat, p_val = ttest_ind(groep_derby, groep_algemeen, equal_var=False)
print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Derby/rivaliteitswedstrijden hebben een SIGNIFICANT effect op het aantal schoten.")
else:
    print("❌ Conclusie: Derby/rivaliteitswedstrijden lijken GEEN significant effect te hebben op het aantal schoten.")

Gemiddeld aantal schoten bij derby/rivaliteit: 24.77
Algemeen gemiddeld aantal schoten: 24.80
👉 Bij derby/rivaliteitswedstrijden vallen gemiddeld minder schoten per wedstrijd.



p-waarde: 0.98687
❌ Conclusie: Derby/rivaliteitswedstrijden lijken GEEN significant effect te hebben op het aantal schoten.


### Weekend → resultaat

In [58]:
df['thuis_wint'] = (df['resultaat_voltijd'] == 'H').astype(int)

df.groupby('is_weekend')['thuis_wint'].mean().plot(kind='bar')
plt.title("Thuiswinst kans: weekend vs niet-weekend")
plt.ylabel("Kans op thuiswinst")
plt.show()

df['uit_wint'] = (df['resultaat_voltijd'] == 'A').astype(int)

df.groupby('is_weekend')['uit_wint'].mean().plot(kind='bar')
plt.title("Uitwinst kans: weekend vs niet-weekend")
plt.ylabel("Kans op uitwinst")
plt.show()

df['gelijkspel'] = (df['resultaat_voltijd'] == 'D').astype(int)

df.groupby('is_weekend')['gelijkspel'].mean().plot(kind='bar')
plt.title("Gelijkspel kans: weekend vs niet-weekend")
plt.ylabel("Kans op gelijkspel")
plt.show()

### Als team A 10 plaatsen boven team B staat in het klassement: hoe groot is volgens u de kans op een overwinning van team A?  Percentage (0 - 100)

In [59]:
# Filter alle wedstrijden waar het rangverschil 10 of -10 is
df_10 = df[df['rank_diff'].abs() == 10].copy()

# Bepaal het hoger gerangschikte team
def hoger_team(row):
    return row['thuis_team'] if row['rank_diff'] < 0 else row['uit_team']

df_10['hoger_team'] = df_10.apply(hoger_team, axis=1)

# Bepaal wie echt won
def echte_winnaar(row):
    if row['resultaat_voltijd'] == 'H':
        return row['thuis_team']
    elif row['resultaat_voltijd'] == 'A':
        return row['uit_team']
    else:
        return 'gelijk'

df_10['echte_winnaar'] = df_10.apply(echte_winnaar, axis=1)

# Check of hoger team gewonnen heeft
df_10['hoger_team_wint'] = df_10['hoger_team'] == df_10['echte_winnaar']

# Bereken percentage van alle wedstrijden
totaal = len(df_10)
aantal_hoger_wint = df_10['hoger_team_wint'].sum()
percentage = (aantal_hoger_wint / totaal) * 100

print(f"Aantal wedstrijden met rangverschil ±10: {totaal}")
print(f"Aantal keer dat het hoger gerangschikte team won: {aantal_hoger_wint}")
print(f"Percentage dat het hoger gerangschikte team won: {percentage:.2f}%")

# Maak een nieuwe kolom met drie categorieën: 'Hoger team wint', 'Lager team wint', 'Gelijk'
def uitslag_category(row):
    if row['echte_winnaar'] == 'gelijk':
        return 'Gelijk'
    elif row['hoger_team_wint']:
        return 'Hoger team wint'
    else:
        return 'Lager team wint'

df_10['uitslag_cat'] = df_10.apply(uitslag_category, axis=1)

# Tel het aantal per categorie
uitslag_counts = df_10['uitslag_cat'].value_counts().reset_index()
uitslag_counts.columns = ['Uitslag', 'Aantal']

# Bereken percentage
uitslag_counts['Percentage'] = (uitslag_counts['Aantal'] / uitslag_counts['Aantal'].sum()) * 100

# Plot een mooie staafgrafiek
plt.figure(figsize=(8,5))
sns.barplot(data=uitslag_counts, x='Uitslag', y='Percentage', palette='coolwarm')
plt.title('Verdeling uitslag bij wedstrijden met rangverschil 10')
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
for i, row in uitslag_counts.iterrows():
    plt.text(i, row['Percentage']+1, f"{row['Percentage']:.1f}%", ha='center', fontsize=10)
plt.show()

Aantal wedstrijden met rangverschil ±10: 109
Aantal keer dat het hoger gerangschikte team won: 86
Percentage dat het hoger gerangschikte team won: 78.90%


### Stel dat een team 3 keer op rij verloren heeft, hoe groot is volgens u de kans op een 4e nederlaag?  Percentage (0 - 100)

In [60]:
# Combineer thuis- en uitteam in één dataframe
teams = pd.concat([
    df[['thuis_team','resultaat_voltijd','wedstrijd_datum']].rename(
        columns={'thuis_team':'team','resultaat_voltijd':'resultaat'}),
    df[['uit_team','resultaat_voltijd','wedstrijd_datum']].rename(
        columns={'uit_team':'team','resultaat_voltijd':'resultaat'})
])

# Sorteer per team en datum
teams = teams.sort_values(['team','wedstrijd_datum']).reset_index(drop=True)

# Maak een kolom 'verlies' (1 = verlies, 0 = niet verloren)
# Verlies = H als thuis, A als uit
teams['verlies'] = teams['resultaat'].map({'H':1, 'A':1, 'D':0})

# Bereken rolling window van 3 opeenvolgende verliezen
teams['verlies_3_reeks'] = teams.groupby('team')['verlies'].rolling(window=3).sum().reset_index(level=0, drop=True)

# Kijk of de volgende wedstrijd ook verlies was
teams['volgende_verlies'] = teams.groupby('team')['verlies'].shift(-1)

# Filter gevallen waar team 3 keer op rij verloor
teams_3verlies = teams[teams['verlies_3_reeks'] == 3]

# Percentage dat 4e verlies volgde
totaal = len(teams_3verlies)
aantal_vierde = teams_3verlies['volgende_verlies'].sum()
percentage_vierde = (aantal_vierde / totaal) * 100

print(f"Aantal gevallen met 3 opeenvolgende verliezen: {totaal}")
print(f"Aantal keer dat een 4e verlies volgde: {aantal_vierde}")
print(f"Percentage dat een 4e verlies volgde: {percentage_vierde:.2f}%")

Aantal gevallen met 3 opeenvolgende verliezen: 1731
Aantal keer dat een 4e verlies volgde: 1315.0
Percentage dat een 4e verlies volgde: 75.97%


### Avondmatch → doelpunten

In [61]:
sns.boxplot(x=df['avond_match'], y=df['totaal_doelpunten'])
plt.title("Avondmatch vs doelpunten")
plt.show()

df.groupby('avond_match')['totaal_doelpunten'].mean().plot(kind='bar')
plt.title("Gemiddelde doelpunten avond vs niet")
plt.show()

### Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?

In [62]:
# ===============================
# 1. Subset: specifieke combo
# ===============================
combo_df = df[df["speelstijl_combo"] == "Aanvallend_vs_Verdedigend"].copy()

# ===============================
# 2. Gemiddelden
# ===============================
gemiddelde_combo = combo_df["totaal_doelpunten"].mean()
gemiddelde_algemeen = df["totaal_doelpunten"].mean()

print(f"Gemiddeld aantal doelpunten (Aanvallend vs Verdedigend): {gemiddelde_combo:.2f}")
print(f"Algemeen gemiddeld aantal doelpunten: {gemiddelde_algemeen:.2f}")

# ===============================
# 3. Interpretatie
# ===============================
if gemiddelde_combo > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_combo < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij deze speelstijl-combinatie vallen gemiddeld {conclusie} doelpunten per wedstrijd.")

# ===============================
# 4. Visualisatie
# ===============================
plt.figure(figsize=(10,6))

sns.boxplot(
    x=df["speelstijl_combo"].apply(lambda x: "Aanvallend_vs_Verdedigend" if x == "Aanvallend_vs_Verdedigend" else "Andere"),
    y=df["totaal_doelpunten"]
)

plt.title("Doelpunten per speelstijl-combinatie")
plt.xlabel("Speelstijl")
plt.ylabel("Totaal doelpunten")
plt.show()

# ===============================
# 5. Statistische test
# ===============================
groep_combo = combo_df["totaal_doelpunten"].dropna()
groep_rest = df[df["speelstijl_combo"] != "Aanvallend_vs_Verdedigend"]["totaal_doelpunten"].dropna()

t_stat, p_val = ttest_ind(groep_combo, groep_rest, equal_var=False)

print(f"\np-waarde: {p_val:.5f}")

if p_val < 0.05:
    print("✅ Conclusie: Speelstijl-combinatie heeft een SIGNIFICANT effect op doelpunten.")
else:
    print("❌ Conclusie: Speelstijl-combinatie heeft GEEN significant effect op doelpunten.")

Gemiddeld aantal doelpunten (Aanvallend vs Verdedigend): 3.19
Algemeen gemiddeld aantal doelpunten: 2.86
👉 Bij deze speelstijl-combinatie vallen gemiddeld meer doelpunten per wedstrijd.



p-waarde: 0.00686
✅ Conclusie: Speelstijl-combinatie heeft een SIGNIFICANT effect op doelpunten.


### Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?

In [63]:
# Subset van derby/rivaliteitswedstrijden
derby_subset = df[df['derby/rivaal'] == 1].copy()

# Gemiddelden berekenen
gemiddelde_derby = derby_subset['totaal_kaarten'].mean()
gemiddelde_algemeen = df['totaal_kaarten'].mean()

print(f"Gemiddeld aantal kaarten bij derby/rivaliteit: {gemiddelde_derby:.2f}")
print(f"Algemeen gemiddeld aantal kaarten: {gemiddelde_algemeen:.2f}")

# Interpretatie
if gemiddelde_derby > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_derby < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij derby/rivaliteitswedstrijden worden gemiddeld {conclusie} kaarten gegeven.")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(
    x=df['derby/rivaal'].map({0:'Geen derby', 1:'Derby/Rivaal'}),
    y=df['totaal_kaarten']
)
plt.title("Aantal kaarten per wedstrijd: Derby vs Geen Derby")
plt.xlabel("Wedstrijdtype")
plt.ylabel("Totaal aantal kaarten")
plt.show()

# Statistische test (t-test)
groep_derby = derby_subset['totaal_kaarten'].dropna()
groep_algemeen = df['totaal_kaarten'].dropna()

t_stat, p_val = ttest_ind(groep_derby, groep_algemeen, equal_var=False)

print(f"\nT-test p-value: {p_val:.5f}")
if p_val < 0.05:
    print("✅ Conclusie: Derby/rivaliteitswedstrijden hebben een SIGNIFICANTE invloed op het aantal kaarten.")
else:
    print("❌ Conclusie: Derby/rivaliteitswedstrijden lijken GEEN significant effect te hebben op het aantal kaarten.")

Gemiddeld aantal kaarten bij derby/rivaliteit: 4.32
Algemeen gemiddeld aantal kaarten: 4.13
👉 Bij derby/rivaliteitswedstrijden worden gemiddeld meer kaarten gegeven.



T-test p-value: 0.71765
❌ Conclusie: Derby/rivaliteitswedstrijden lijken GEEN significant effect te hebben op het aantal kaarten.


### Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?

In [64]:
# Subset van degradatiewedstrijden
degradatie_subset = df[df['degradatie_match'] == 1].copy()

# Gemiddelden berekenen
gemiddelde_degradatie = degradatie_subset['totaal_kaarten'].mean()
gemiddelde_algemeen = df['totaal_kaarten'].mean()

print(f"Gemiddeld aantal kaarten bij degradatiewedstrijden: {gemiddelde_degradatie:.2f}")
print(f"Algemeen gemiddeld aantal kaarten: {gemiddelde_algemeen:.2f}")

# Interpretatie
if gemiddelde_degradatie > gemiddelde_algemeen:
    conclusie = "meer"
elif gemiddelde_degradatie < gemiddelde_algemeen:
    conclusie = "minder"
else:
    conclusie = "hetzelfde"

print(f"👉 Bij degradatiewedstrijden worden gemiddeld {conclusie} kaarten gegeven.")

# Visualisatie
plt.figure(figsize=(8,6))
sns.boxplot(
    x=df['degradatie_match'].map({0:'Geen degradatiewedstrijd', 1:'Degradatiewedstrijd'}),
    y=df['totaal_kaarten']
)
plt.title("Aantal kaarten per wedstrijd: Degradatie vs Geen degradatie")
plt.xlabel("Wedstrijdtype")
plt.ylabel("Totaal aantal kaarten")
plt.show()

# Statistische test (t-test)
groep_degradatie = degradatie_subset['totaal_kaarten'].dropna()
groep_algemeen = df['totaal_kaarten'].dropna()

t_stat, p_val = ttest_ind(groep_degradatie, groep_algemeen, equal_var=False)

print(f"\nT-test p-value: {p_val:.5f}")
if p_val < 0.05:
    print("✅ Conclusie: Degradatiewedstrijden hebben een SIGNIFICANTE invloed op het aantal kaarten.")
else:
    print("❌ Conclusie: Degradatiewedstrijden lijken GEEN significant effect te hebben op het aantal kaarten.")

Gemiddeld aantal kaarten bij degradatiewedstrijden: 4.88
Algemeen gemiddeld aantal kaarten: 4.13
👉 Bij degradatiewedstrijden worden gemiddeld meer kaarten gegeven.



T-test p-value: 0.28235
❌ Conclusie: Degradatiewedstrijden lijken GEEN significant effect te hebben op het aantal kaarten.


In [65]:
result_dist = df['resultaat_voltijd'].value_counts(normalize=True)

print(result_dist)

result_dist.plot(kind='bar')
plt.title("Werkelijke kans op resultaat")
plt.ylabel("Proportie")
plt.show()

resultaat_voltijd
H    0.433200
A    0.323031
D    0.243769
Name: proportion, dtype: float64


In [66]:
print("Gemiddelde goals:")
print(df[['doelpunten_thuis_voltijd','doelpunten_uit_voltijd']].mean())

sns.histplot(df['doelpunten_thuis_voltijd'], kde=True)
plt.title("Goals thuis")
plt.show()

sns.histplot(df['doelpunten_uit_voltijd'], kde=True)
plt.title("Goals uit")
plt.show()

Gemiddelde goals:
doelpunten_thuis_voltijd    1.568295
doelpunten_uit_voltijd      1.290628
dtype: float64


### Data reset

In [67]:
df = main_features.copy()

# Print alle kolommen van je dataframe
print("Aantal kolommen:", len(df.columns))
print(df.columns.tolist())

Aantal kolommen: 66
['wedstrijd_datum', 'thuis_team', 'uit_team', 'doelpunten_thuis_voltijd', 'doelpunten_uit_voltijd', 'resultaat_voltijd', 'schoten_thuis', 'schoten_uit', 'schoten_op_doel_thuis', 'schoten_op_doel_uit', 'overtredingen_thuis', 'overtredingen_uit', 'corners_thuis', 'corners_uit', 'gele_kaarten_thuis', 'gele_kaarten_uit', 'rode_kaarten_thuis', 'rode_kaarten_uit', 'relatieve_luchtvochtigheid_pct', 'windstoten_m_s', 'bewolking_pct', 'weercode', 'luchtdruk_hpa', 'temperatuur_gem_c', 'neerslag_som_mm', 'windsnelheid_gem_m_s', 'windstoten_max_m_s', 'thuis_stadion_capaciteit', 'seizoen', 'thuis_rank', 'uit_rank', 'titel_match', 'degradatie_match', 'derby/rivaal', 'home_shots_for', 'home_goals_for', 'home_corners_for', 'home_shots_against', 'home_goals_against', 'away_shots_for', 'away_goals_for', 'away_corners_for', 'away_shots_against', 'away_goals_against', 'home_style_score', 'away_style_score', 'thuis_speelstijl', 'uit_speelstijl', 'speelstijl_combo', 'thuis_form_5', 'uit_

## 7. Modellen

- Keuze model (logistische regressie, classificatie)
- Features en target
- Train-test split
- Resultaten evaluatie

In [68]:
df['missing_form'] = df['form_diff'].isna().astype(int)
df['form_diff'] = df['form_diff'].fillna(0)
df['thuis_form_5'] = df['thuis_form_5'].fillna(0)
df['uit_form_5'] = df['uit_form_5'].fillna(0)
df['thuisvoordeel'] = 1

### Model voor eindresultaat

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. TARGET
# ===============================
le = LabelEncoder()
df["resultaat_encoded"] = le.fit_transform(df["resultaat_voltijd"])
y = df["resultaat_encoded"]

# ===============================
# 3. FEATURES
# ===============================
features = [ 
    # team strength 
    "rank_diff", "form_diff", "h2h_resultaat", 
    # match context 
    "seizoen", "weekday", "is_weekend", "avond_match", "derby/rivaal", 
    # # weather 
    "temperatuur_gem_c", "windstoten_m_s", "bewolking_pct", "neerslag_som_mm", "windsnelheid_gem_m_s", 
    # stadium # style 
    "speelstijl_combo" ]

X = pd.get_dummies(df[features], drop_first=True)

# ===============================
# 4. TIME SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# ===============================
# 5. CV
# ===============================
tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 6. MODELS + PARAM GRIDS
# ===============================
models = {
    "LogReg": (
        LogisticRegression(max_iter=1000, solver="lbfgs"),
        {
            "C": [0.01, 0.1, 1, 10]
        }
    ),

    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [200, 500],
            "max_depth": [5, 10, None]
        }
    ),

    "GradientBoosting": (
        GradientBoostingClassifier(),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            random_state=42
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0]
        }
    )
}

# ===============================
# 7. TRAINING + GRIDSEARCH
# ===============================
results = []
best_models = {}

for name, (model, params) in models.items():

    grid = GridSearchCV(
        model,
        params,
        scoring="neg_log_loss",
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best = grid.best_estimator_

    preds = best.predict(X_test)
    probs = best.predict_proba(X_test)

    results.append({
        "model": name,
        "best_params": grid.best_params_,
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds, average="macro"),
        "precision": precision_score(y_test, preds, average="macro"),
        "recall": recall_score(y_test, preds, average="macro"),
        "log_loss": log_loss(y_test, probs)
    })

    best_models[name] = best

    print("\n======================")
    print(name)
    print(classification_report(y_test, preds, target_names=le.classes_))

# ===============================
# 8. BEST MODEL SELECTIE
# ===============================
results_df = pd.DataFrame(results)
best_model_name = results_df.sort_values("log_loss").iloc[0]["model"]
best_model = best_models[best_model_name]

print("\n==============================")
print("BEST MODEL:", best_model_name)
print("==============================")
print(results_df.sort_values("log_loss"))

# ===============================
# 9. FINAL METRICS BEST MODEL
# ===============================
final_preds = best_model.predict(X_test)
final_probs = best_model.predict_proba(X_test)

print("\nFINAL EVALUATION BEST MODEL")
print("Accuracy:", accuracy_score(y_test, final_preds))
print("F1:", f1_score(y_test, final_preds, average="macro"))
print("Log Loss:", log_loss(y_test, final_probs))

# ===============================
# 10. FEATURE IMPORTANCE BEST MODEL
# ===============================

import pandas as pd
import matplotlib.pyplot as plt

print("\n==============================")
print("FEATURE IMPORTANCE BEST MODEL")
print("==============================")

if hasattr(best_model, "feature_importances_"):

    # tree-based models (RF, XGB, GB)
    fi = pd.DataFrame({
        "feature": X.columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

elif hasattr(best_model, "coef_"):

    # logistic regression
    fi = pd.DataFrame({
        "feature": X.columns,
        "importance": np.abs(best_model.coef_[0])
    }).sort_values("importance", ascending=False)

else:
    fi = None
    print("Geen feature importance beschikbaar voor dit model.")

if fi is not None:

    print(fi.head(20))

    # ===============================
    # VISUALISATIE TOP FEATURES
    # ===============================
    plt.figure(figsize=(10,6))
    plt.barh(fi["feature"].head(15)[::-1], fi["importance"].head(15)[::-1])
    plt.title("Top 15 Feature Importance - Best Model")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

### Voorspelling eindresultaten wedstrijden

In [70]:
# ===============================
# PREDICTIE OP df_predict
# ===============================

X_predict = df_predict[features]
X_predict = pd.get_dummies(X_predict, drop_first=True)

# align columns
X_predict = X_predict.reindex(columns=X_train.columns, fill_value=0)

# best model fix
results_df_models = pd.DataFrame(results)
best_model_name = results_df_models.sort_values("accuracy", ascending=False).iloc[0]["model"]
best_model = best_models[best_model_name]

print("BEST MODEL USED FOR PREDICTION:", best_model_name)

# predict
probs = best_model.predict_proba(X_predict)
preds = best_model.predict(X_predict)

labels = le.inverse_transform(preds)

# results
results_df = pd.DataFrame({
    "datum": df_predict["wedstrijd_datum"].values,
    "thuis_team": df_predict["thuis_team"].values,
    "uit_team": df_predict["uit_team"].values,
    "voorspelling": labels,
    "P_A": probs[:, 0],
    "P_D": probs[:, 1],
    "P_H": probs[:, 2],
})

results_df["zekerheid"] = results_df[["P_A", "P_D", "P_H"]].max(axis=1)

results_df = results_df.sort_values("zekerheid", ascending=False)

display(results_df)

BEST MODEL USED FOR PREDICTION: LogReg


In [98]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. FEATURES
# ===============================
features = [
    "rank_diff",
    "form_diff",
    "h2h_resultaat",
    "is_weekend",
    "avond_match",
    "titel_match",
    "degradatie_match",
    "temperatuur_gem_c",
    "neerslag_som_mm",
    "regen",
    "harde_regen",
    "thuis_stadion_capaciteit",
    "speelstijl_combo"
]

X = df[features]
X = pd.get_dummies(X, drop_first=True)

# ===============================
# 3. SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 4. MODELS + GRID
# ===============================
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [200, 500],
            "max_depth": [5, 10, None]
        }
    ),
    "XGBoost": (
        XGBRegressor(objective="reg:squarederror", random_state=42),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0]
        }
    )
}

# ===============================
# 5. TARGETS
# ===============================
targets = {
    "home_goals": "doelpunten_thuis_voltijd",
    "away_goals": "doelpunten_uit_voltijd"
}

best_models = {}

# ===============================
# 6. TRAIN MODELS
# ===============================
for name, target_col in targets.items():

    print("\n==============================")
    print(f"MODEL: {name}")
    print("==============================")

    y = df[target_col]

    y_train = y.iloc[:split]
    y_test = y.iloc[split:]

    best_score = -np.inf
    best_model = None
    best_params = None

    for model_name, (model, params) in models.items():

        grid = GridSearchCV(
            model,
            params,
            scoring="neg_mean_absolute_error",
            cv=tscv,
            n_jobs=-1
        )

        grid.fit(X_train, y_train)

        preds = grid.best_estimator_.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        print(f"{model_name} -> MAE: {mae:.3f} | R2: {r2:.3f}")

        if r2 > best_score:
            best_score = r2
            best_model = grid.best_estimator_
            best_params = grid.best_params_

    best_models[name] = best_model

    print("\nBEST MODEL:", type(best_model).__name__)
    print("BEST PARAMS:", best_params)

# ===============================
# 7. PREDICT df_predict
# ===============================

X_predict = df_predict[features]
X_predict = pd.get_dummies(X_predict, drop_first=True)
X_predict = X_predict.reindex(columns=X_train.columns, fill_value=0)

home_model = best_models["home_goals"]
away_model = best_models["away_goals"]

df_predict_results = df_predict[["wedstrijd_datum", "thuis_team", "uit_team"]].copy()

df_predict_results["pred_goals_thuis"] = home_model.predict(X_predict)
df_predict_results["pred_goals_uit"] = away_model.predict(X_predict)

df_predict_results["pred_totaal_goals"] = (
    df_predict_results["pred_goals_thuis"] +
    df_predict_results["pred_goals_uit"]
)

print("\n==============================")
print("GOAL PREDICTIONS")
print("==============================")

display(df_predict_results)


MODEL: home_goals
RandomForest -> MAE: 0.924 | R2: 0.098
XGBoost -> MAE: 0.943 | R2: 0.060

BEST MODEL: RandomForestRegressor
BEST PARAMS: {'max_depth': 5, 'n_estimators': 500}

MODEL: away_goals
RandomForest -> MAE: 0.856 | R2: 0.010
XGBoost -> MAE: 0.853 | R2: 0.005

BEST MODEL: RandomForestRegressor
BEST PARAMS: {'max_depth': 5, 'n_estimators': 500}

GOAL PREDICTIONS


### Model voor aantal kaarten

In [99]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. TARGET
# ===============================
y = df["totaal_kaarten"]

# ===============================
# 3. FEATURES
# ===============================
features = [
    "abs_rank_diff",
    "form_diff",
    "h2h_resultaat",
    "seizoen",
    "is_weekend",
    "avond_match",
    "titel_match",
    "degradatie_match",
    "derby/rivaal",
    "temperatuur_gem_c",
    "neerslag_som_mm",
    "regen",
    "harde_regen",
    "thuis_stadion_capaciteit",
    "speelstijl_combo"
]

X = df[features]
X = pd.get_dummies(X, drop_first=True)

# ===============================
# 4. TIME SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# ===============================
# 5. CV
# ===============================
tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 6. MODELS + GRID
# ===============================
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [200, 500],
            "max_depth": [5, 10, None]
        }
    ),

    "GradientBoosting": (
        GradientBoostingRegressor(),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5]
        }
    ),

    "XGBoost": (
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0]
        }
    )
}

# ===============================
# 7. TRAINING + GRIDSEARCH
# ===============================
results = []
best_models = {}

for name, (model, params) in models.items():

    grid = GridSearchCV(
        model,
        params,
        scoring="r2",
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    preds = best.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "model": name,
        "mae": mae,
        "r2": r2,
        "best_params": grid.best_params_
    })

    best_models[name] = best

    print("\n======================")
    print(name)
    print("MAE:", round(mae, 3))
    print("R2:", round(r2, 3))
    print("Best params:", grid.best_params_)

# ===============================
# 8. BEST MODEL
# ===============================
results_df = pd.DataFrame(results).sort_values("r2", ascending=False)

best_model_name = results_df.iloc[0]["model"]
best_model = best_models[best_model_name]

print("\nBEST MODEL:", best_model_name)

# ===============================
# 9. FEATURE IMPORTANCE
# ===============================
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTOP FEATURES:")
print(fi.head(20))

# ===============================
# 10. GROUPING (THESIS READY)
# ===============================
regen_features = ["neerslag_som_mm", "regen", "harde_regen"]
tijdstip = ["is_weekend", "avond_match"]

def group_feature(name):

    if "h2h_resultaat" in name:
        return "h2h_resultaat"

    if "speelstijl_combo" in name:
        return "speelstijl_combo"

    if "capaciteit" in name:
        return "stadion_capaciteit"

    if name in regen_features:
        return "neerslag"

    if "abs_rank_diff" in name:
        return "abs_rank_diff"

    if "form_diff" in name:
        return "form_diff"

    if "titel_match" in name:
        return "titel_match"

    if "degradatie_match" in name:
        return "degradatie_match"

    if name in tijdstip:
        return "tijdstip"

    return name

fi["group"] = fi["feature"].apply(group_feature)

grouped = fi.groupby("group")["importance"].sum().reset_index()
grouped = grouped.sort_values("importance", ascending=False)

print("\nGROUPED IMPORTANCE:")
print(grouped)

# ===============================
# 11. VISUALISATIE
# ===============================
plt.figure(figsize=(10,6))
sns.barplot(data=grouped, x="importance", y="group")
plt.title("Feature Importance - Total Cards")
plt.tight_layout()
plt.show()

# ===============================
# 12. SHAP ANALYSIS
# ===============================
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

### Model voor schoten

In [73]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. TARGET
# ===============================
y = df["totaal_schoten"]

# ===============================
# 3. FEATURES
# ===============================
features = [
    "abs_rank_diff",
    "form_diff",
    "h2h_resultaat",
    "titel_match",
    "degradatie_match",
    "derby/rivaal",
    "is_weekend",
    "avond_match",
    "temperatuur_gem_c",
    "neerslag_som_mm",
    "regen",
    "harde_regen",
    "thuis_stadion_capaciteit",
    "speelstijl_combo"
]

X = df[features]
X = pd.get_dummies(X, drop_first=True)

# ===============================
# 4. SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

# ===============================
# 5. TIME SERIES CV
# ===============================
tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 6. MODEL + GRID
# ===============================
model = XGBRegressor(objective="reg:squarederror", random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

# ===============================
# 7. BEST MODEL
# ===============================
best_model = grid.best_estimator_

print("\nBEST PARAMETERS:")
print(grid.best_params_)

# ===============================
# 8. EVALUATION
# ===============================
preds = best_model.predict(X_test)

print("\nMAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

# ===============================
# 9. FEATURE IMPORTANCE
# ===============================
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTOP FEATURES:")
print(fi.head(20))

# ===============================
# 10. GROUPED IMPORTANCE
# ===============================
neerslag_features = ["neerslag_som_mm", "regen", "harde_regen"]

tijdstip_features = ["is_weekend", "avond_match"]

def group_feature(name):

    if "abs_rank_diff" in name:
        return "abs_rank_diff"

    if "form_diff" in name:
        return "form_diff"

    if "h2h_resultaat" in name:
        return "h2h_resultaat"

    if name in neerslag_features:
        return "neerslag"

    if "titel_match" in name:
        return "titel_match"

    if "degradatie_match" in name:
        return "degradatie_match"

    if "speelstijl_combo" in name:
        return "speelstijl_combo"
    
    if name in tijdstip_features:
        return "tijdstip"

    return name


fi["group"] = fi["feature"].apply(group_feature)

grouped = fi.groupby("group")["importance"].sum().reset_index()
grouped = grouped.sort_values("importance", ascending=False)

print("\nGROUPED IMPORTANCE:")
print(grouped)

# ===============================
# 11. VISUALISATIE
# ===============================
plt.figure(figsize=(10,6))

sns.barplot(
    data=grouped,
    x="importance",
    y="group"
)

plt.title("Feature Importance - Total Shots (Tuned XGBoost)")
plt.tight_layout()
plt.show()

# ===============================
# 12. SHAP ANALYSIS
# ===============================
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

### Model voor aantal doelpunten

In [74]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. TARGET
# ===============================
y = df["totaal_doelpunten"]

# ===============================
# 3. FEATURES
# ===============================
features = [
    # strength
    "abs_rank_diff",
    "form_diff",
    "h2h_resultaat",

    # context
    "titel_match",
    "degradatie_match",
    "is_weekend",
    "avond_match",

    # weather
    "temperatuur_gem_c",
    "neerslag_som_mm",
    "regen",
    "harde_regen",

    # stadium
    "thuis_stadion_capaciteit",

    # style
    "speelstijl_combo"
]

X = df[features]
X = pd.get_dummies(X, drop_first=True)

# ===============================
# 4. SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# ===============================
# 5. TIME CV
# ===============================
tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 6. MODELS + GRIDSEARCH
# ===============================
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [200, 500],
            "max_depth": [5, 10, None]
        }
    ),

    "GradientBoosting": (
        GradientBoostingRegressor(),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5]
        }
    ),

    "XGBoost": (
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0]
        }
    )
}

# ===============================
# 7. TRAINING
# ===============================
results = []
best_models = {}

for name, (model, params) in models.items():

    grid = GridSearchCV(
        model,
        params,
        scoring="r2",
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    preds = best.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "model": name,
        "mae": mae,
        "r2": r2,
        "best_params": grid.best_params_
    })

    best_models[name] = best

    print("\n======================")
    print(name)
    print("MAE:", round(mae, 3))
    print("R2:", round(r2, 3))
    print("Best params:", grid.best_params_)

# ===============================
# 8. BEST MODEL
# ===============================
results_df = pd.DataFrame(results).sort_values("r2", ascending=False)

best_model_name = results_df.iloc[0]["model"]
best_model = best_models[best_model_name]

print("\nBEST MODEL:", best_model_name)

# ===============================
# 9. FEATURE IMPORTANCE
# ===============================
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTOP FEATURES:")
print(fi.head(20))

# ===============================
# 10. GROUPING (THESIS READY)
# ===============================
neerslag_features = ["neerslag_som_mm", "regen", "harde_regen"]
tijdstip = ["is_weekend", "avond_match"]

def group_feature(name):

    if "h2h_resultaat" in name:
        return "h2h_resultaat"

    if "speelstijl_combo" in name:
        return "speelstijl_combo"

    if "capaciteit" in name:
        return "stadion_capaciteit"

    if name in neerslag_features:
        return "neerslag"

    if "abs_rank_diff" in name:
        return "abs_rank_diff"

    if "form_diff" in name:
        return "form_diff"

    if "titel_match" in name:
        return "titel_match"

    if "degradatie_match" in name:
        return "degradatie_match"

    if name in tijdstip:
        return "tijdstip"

    return name


fi["group"] = fi["feature"].apply(group_feature)

grouped = fi.groupby("group")["importance"].sum().reset_index()
grouped = grouped.sort_values("importance", ascending=False)

print("\nGROUPED IMPORTANCE (GOALS):")
print(grouped)

# ===============================
# 11. VISUALISATIE
# ===============================
plt.figure(figsize=(10,6))
sns.barplot(data=grouped, x="importance", y="group")
plt.title("Feature Importance - Total Goals")
plt.tight_layout()
plt.show()

# ===============================
# 12. SHAP ANALYSIS
# ===============================
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

### Model voor aantal overtredingen

In [75]:
# ===============================
# 1. SORT DATA
# ===============================
df = df.sort_values("wedstrijd_datum").reset_index(drop=True)

# ===============================
# 2. TARGET
# ===============================
y = df["totaal_overtredingen"]

# ===============================
# 3. FEATURES
# ===============================
features = [
    # strength
    "abs_rank_diff",
    "form_diff",
    "h2h_resultaat",

    # context
    "titel_match",
    "degradatie_match",
    "is_weekend",
    "avond_match",
    "derby/rivaal",

    # weather
    "temperatuur_gem_c",
    "neerslag_som_mm",
    "regen",
    "harde_regen",

    # stadium
    "thuis_stadion_capaciteit",

    # style
    "speelstijl_combo"
]

X = df[features]
X = pd.get_dummies(X, drop_first=True)

# ===============================
# 4. SPLIT
# ===============================
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# ===============================
# 5. TIME CV
# ===============================
tscv = TimeSeriesSplit(n_splits=5)

# ===============================
# 6. MODELS + GRIDSEARCH
# ===============================
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            "n_estimators": [200, 500],
            "max_depth": [5, 10, None]
        }
    ),

    "GradientBoosting": (
        GradientBoostingRegressor(),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5]
        }
    ),

    "XGBoost": (
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0]
        }
    )
}

# ===============================
# 7. TRAINING
# ===============================
results = []
best_models = {}

for name, (model, params) in models.items():

    grid = GridSearchCV(
        model,
        params,
        scoring="r2",
        cv=tscv,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    preds = best.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "model": name,
        "mae": mae,
        "r2": r2,
        "best_params": grid.best_params_
    })

    best_models[name] = best

    print("\n======================")
    print(name)
    print("MAE:", round(mae, 3))
    print("R2:", round(r2, 3))
    print("Best params:", grid.best_params_)

# ===============================
# 8. BEST MODEL
# ===============================
results_df = pd.DataFrame(results).sort_values("r2", ascending=False)

best_model_name = results_df.iloc[0]["model"]
best_model = best_models[best_model_name]

print("\nBEST MODEL:", best_model_name)

# ===============================
# 9. FEATURE IMPORTANCE
# ===============================
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTOP FEATURES (FOULS):")
print(fi.head(20))

# ===============================
# 10. GROUPING (THESIS READY)
# ===============================
neerslag_features = ["neerslag_som_mm", "regen", "harde_regen"]
tijdstip = ["is_weekend", "avond_match"]

def group_feature(name):

    if "h2h_resultaat" in name:
        return "h2h_resultaat"

    if "speelstijl_combo" in name:
        return "speelstijl_combo"

    if "capaciteit" in name:
        return "stadion_capaciteit"

    if name in neerslag_features:
        return "neerslag"

    if "rank_diff" in name:
        return "rank_diff"

    if "form_diff" in name:
        return "form_diff"

    if "titel_match" in name:
        return "titel_match"

    if "degradatie_match" in name:
        return "degradatie_match"

    if "derby" in name:
        return "derby"

    if name in tijdstip:
        return "tijdstip"

    return name


fi["group"] = fi["feature"].apply(group_feature)

grouped = fi.groupby("group")["importance"].sum().reset_index()
grouped = grouped.sort_values("importance", ascending=False)

print("\nGROUPED IMPORTANCE (FOULS):")
print(grouped)

# ===============================
# 11. VISUALISATIE
# ===============================
plt.figure(figsize=(10,6))
sns.barplot(data=grouped, x="importance", y="group")
plt.title("Feature Importance - Total Fouls")
plt.tight_layout()
plt.show()

# ===============================
# 12. SHAP ANALYSIS
# ===============================
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

## 8. Model voor doelpunten

- Keuze model (Poisson regression, regressie)
- Features en target
- Resultaten evaluatie

## 9. Model evaluatie

- Accuracy, log-loss, RMSE, MAE
- Grafieken van voorspelling vs echte resultaten

## 10. Expertdata

### CSV (enquête) inladen

In [76]:
enquete_df = pd.read_csv('../data/experts_enquete.csv')

### Kolommen hernoemen

In [77]:
rename_map = {
    "Kans op thuiswinst  (0 - 100)": "Kans_thuis_RSCvCercle",
    "Kans op gelijkspel  (0 - 100)": "Kans_gelijk_RSCvCercle",
    "Kans op uitwinst  (0 - 100)": "Kans_uit_RSCvCercle",
    "Verwacht aantal doelpunten thuisteam?": "Doelpunten_thuis_RSCvCercle",
    "Verwacht aantal doelpunten uitteam?": "Doelpunten_uit_RSCvCercle",
    "Hoe zeker bent u van deze match voorspelling?": "Confidence_RSCvCercle",
    "Kans op thuiswinst  (0 - 100).1": "Kans_thuis_STVVvUSG",
    "Kans op gelijkspel  (0 - 100).1": "Kans_gelijk_STVVvUSG",
    "Kans op uitwinst  (0 - 100).1": "Kans_uit_STVVvUSG",
    "Verwacht aantal doelpunten thuisteam?.1": "Doelpunten_thuis_STVVvUSG",
    "Verwacht aantal doelpunten uitteam?.1": "Doelpunten_uit_STVVvUSG",
    "Hoe zeker bent u van deze match voorspelling?.1": "Confidence_STVVvUSG"
}

enquete_df = enquete_df.rename(columns=rename_map)

### Schalen controleren en omzetten

In [78]:
scale_1_5 = [
    "Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?",
    "In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  ",
    "In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  ",
    "Heeft regen invloed op het aantal doelpunten?  ",
    "In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt? ",
    "Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat? ",
    "Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd? ",
    "In welke mate hebben eerdere confrontaties invloed in uw voorspelling? ",
    "In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd? ",
    "Hoe zeker bent u van deze match voorspelling?",
    "Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?",
    "Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?",
    "Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?"
]

scale_0_100 = [
    "Kans_thuis_RSCvCercle", "Kans_gelijk_RSCvCercle", "Kans_uit_RSCvCercle",
    "Doelpunten_thuis_RSCvCercle", "Doelpunten_uit_RSCvCercle", "Confidence_RSCvCercle",
    "Kans_thuis_STVVvUSG", "Kans_gelijk_STVVvUSG", "Kans_uit_STVVvUSG",
    "Doelpunten_thuis_STVVvUSG", "Doelpunten_uit_STVVvUSG", "Confidence_STVVvUSG",
    "Als team A 10 plaatsen boven team B staat in het klassement: hoe groot is volgens u de kans op een overwinning van team A?  Percentage (0 - 100)",
    "Stel dat een team 3 keer op rij verloren heeft, hoe groot is volgens u de kans op een 4e nederlaag?  Percentage (0 - 100)"
]

# 1-5 schaal: numeriek en foutcontrole
for col in scale_1_5:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce").clip(1,5)

# 0-100 schaal: numeriek, clip 0-100
for col in scale_0_100:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce").clip(0,100)

### Test: Som van kansen voor wedstrijden moet 100 zijn

In [79]:
# Wedstrijd 1: RSC Anderlecht vs Cercle Brugge
enquete_df['kans_som_match1'] = (
    enquete_df['Kans_thuis_RSCvCercle'] +
    enquete_df['Kans_gelijk_RSCvCercle'] +
    enquete_df['Kans_uit_RSCvCercle']
)

# Wedstrijd 2: Sint-Truidense VV vs Union Saint-Gilloise
enquete_df['kans_som_match2'] = (
    enquete_df['Kans_thuis_STVVvUSG'] +
    enquete_df['Kans_gelijk_STVVvUSG'] +
    enquete_df['Kans_uit_STVVvUSG']
)

# Controleer wedstrijd 1
fouten_match1 = enquete_df[enquete_df['kans_som_match1'] != 100]
if not fouten_match1.empty:
    print("Fouten in kans-som voor match 1 (RSCvCercle):")
    print(fouten_match1[['Kans_thuis_RSCvCercle', 'Kans_gelijk_RSCvCercle', 'Kans_uit_RSCvCercle', 'kans_som_match1']])
else:
    print("Alle sommen van kansvragen voor match 1 zijn correct (=100)")

# Controleer wedstrijd 2
fouten_match2 = enquete_df[enquete_df['kans_som_match2'] != 100]
if not fouten_match2.empty:
    print("\nFouten in kans-som voor match 2 (STVVvUSG):")
    print(fouten_match2[['Kans_thuis_STVVvUSG', 'Kans_gelijk_STVVvUSG', 'Kans_uit_STVVvUSG', 'kans_som_match2']])
else:
    print("Alle sommen van kansvragen voor match 2 zijn correct (=100)")

# Verwijder tijdelijke kolommen na controle
enquete_df = enquete_df.drop(columns=['kans_som_match1', 'kans_som_match2'])

Alle sommen van kansvragen voor match 1 zijn correct (=100)
Alle sommen van kansvragen voor match 2 zijn correct (=100)


In [80]:
enquete_df.info()

enquete_df = enquete_df.drop(['Tijdstempel'], axis=1)

enquete_df.head()

### Factorbelang van wedstrijdstatistieken (inclusief mentale aspecten en motivatie)

In [81]:
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)

for col in scale_1_5:
    if col in enquete_df.columns:
        plt.figure()
        sns.countplot(data=enquete_df, x=col, palette="viridis")
        plt.title(f"Verdeling antwoorden: {col}")
        plt.xlabel("Score (1-5)")
        plt.ylabel("Aantal respondenten")
        plt.xticks([0,1,2,3,4], [1,2,3,4,5])
        plt.show()

### Concrete voorspellingen voor geselecteerde wedstrijden

In [82]:
# Wedstrijden
wedstrijden = [
    {
        "prefix": "RSCvCercle",
        "cols_pct": ['Kans_thuis_RSCvCercle', 'Kans_gelijk_RSCvCercle', 'Kans_uit_RSCvCercle'],
        "cols_num": ['Doelpunten_thuis_RSCvCercle', 'Doelpunten_uit_RSCvCercle', 'Confidence_RSCvCercle']
    },
    {
        "prefix": "STVVvUSG",
        "cols_pct": ['Kans_thuis_STVVvUSG', 'Kans_gelijk_STVVvUSG', 'Kans_uit_STVVvUSG'],
        "cols_num": ['Doelpunten_thuis_STVVvUSG', 'Doelpunten_uit_STVVvUSG', 'Confidence_STVVvUSG']
    }
]

for match in wedstrijden:
    # --- Percentage vragen (0-100) ---
    for col in match['cols_pct']:
        if col in enquete_df.columns:
            data = enquete_df[col].clip(0, 100)
            plt.figure(figsize=(6,4))
            sns.violinplot(x=data, inner="quartile", color="skyblue")
            sns.stripplot(x=data, color='black', jitter=True, size=7, alpha=0.7)
            plt.title(f"Verdeling + individuele antwoorden (0-100): {col}")
            plt.xlabel("Percentage")
            plt.show()
    
    # --- Numerieke vragen (doelpunten, confidence) ---
    for col in match['cols_num']:
        if col in enquete_df.columns:
            data = enquete_df[col].dropna()
            plt.figure(figsize=(6,4))
            
            # Gemiddelde voor doelpunten
            if "Doelpunten" in col:
                mean_val = data.mean()
                sns.histplot(data, bins=range(int(data.min()), int(data.max())+2), color="skyblue", kde=False)
                plt.axvline(mean_val, color='red', linestyle='--', label=f'Gemiddelde = {mean_val:.1f}')
                plt.legend()
                plt.title(f"Verdeling doelpunten: {col}")
                plt.xlabel("Aantal doelpunten")
                plt.ylabel("Aantal respondenten")
            
            # Confidence 1-5 schaal
            elif "Confidence" in col:
                sns.countplot(x=data, palette="viridis", order=range(1,6))
                plt.title(f"Verdeling confidence (1-5): {col}")
                plt.xlabel("Score (1-5)")
                plt.ylabel("Aantal respondenten")
            
            plt.show()

### Overige kans vragen

In [83]:
kans_vragen = [
    "Als team A 10 plaatsen boven team B staat in het klassement: hoe groot is volgens u de kans op een overwinning van team A?  Percentage (0 - 100)",
    "Stel dat een team 3 keer op rij verloren heeft, hoe groot is volgens u de kans op een 4e nederlaag?  Percentage (0 - 100)"
]

for col in kans_vragen:
    if col in enquete_df.columns:
        # Clip waarden tussen 0 en 100
        data = enquete_df[col].clip(0, 100)

        plt.figure(figsize=(6,4))
        sns.violinplot(x=data, inner="quartile", color="skyblue")
        sns.stripplot(x=data, color='black', jitter=True, size=7, alpha=0.7)
        plt.title(f"Verdeling + individuele antwoorden: {col}")
        plt.xlabel("Percentage")
        plt.show()

### Ranking van factoren

In [88]:
def borda_score_plot(df, keyword, titel, figsize=(8,5), palette="Blues_d"):
    """
    Bereken Borda scores voor ranking-vragen en plot de resultaten.

    Parameters:
    - df: DataFrame met enquete resultaten
    - keyword: string die in kolomnamen voorkomt, bv. "winst/verlies"
    - titel: titel van de plot
    - figsize: grootte van de plot
    - palette: kleuren voor de barplot
    """
    # Filter kolommen met rankings
    cols = [col for col in df.columns if keyword in col and "[" in col]
    if not cols:
        print(f"Geen kolommen gevonden voor keyword: {keyword}")
        return
    
    ranking = df[cols]
    scores = {}

    for col in ranking.columns:
        # Extract positie uit kolomnaam (bijv. "[1e]" → 1)
        match = re.search(r"\[(\d+)e\]", col)
        if match:
            positie = int(match.group(1))
        else:
            continue
        # Borda punten: 1e = max punten
        punten = len(cols) - positie + 1

        for factor in ranking[col]:
            if pd.notnull(factor):
                scores[factor] = scores.get(factor, 0) + punten

    # Sorteer scores van hoog naar laag
    scores_df = pd.Series(scores).sort_values(ascending=False)

    # Plot
    plt.figure(figsize=figsize)
    sns.barplot(x=scores_df.values, y=scores_df.index, palette=palette)
    plt.title(titel)
    plt.xlabel("Borda score")
    plt.ylabel("Factor")
    plt.show()

    # Extra: Top1 verdeling
    top1_col = cols[0]
    top1_counts = ranking[top1_col].value_counts().sort_values(ascending=False)
    plt.figure(figsize=figsize)
    sns.barplot(x=top1_counts.values, y=top1_counts.index, palette="Greens_d")
    plt.title(f"Welke factor staat het vaakst op plaats 1 ({keyword})")
    plt.xlabel("Aantal keer 1e plaats")
    plt.ylabel("Factor")
    plt.show()

    return scores_df

In [89]:
# Winst/verlies
scores_winst = borda_score_plot(enquete_df, "winst/verlies", "Belangrijkste factoren voor winst/verlies")

# Doelpunten
scores_doelpunten = borda_score_plot(enquete_df, "aantal doelpunten", "Belangrijkste factoren voor aantal doelpunten")

# Kaarten
scores_kaarten = borda_score_plot(enquete_df, "aantal kaarten", "Belangrijkste factoren voor aantal kaarten")

### Intuïtieve causale inschattingen

In [90]:
causale_vragen = [
    "Als het hard regent, zullen er ... kaarten verwacht worden.",
    "Als het hard regent, zullen er ... doelpunten verwacht worden.",
    "Als het thuisteam een aanvallende speelstijl hanteert en het uitteam defensief speelt zullen er gemiddeld ... doelpunten vallen.",
    "Als het erg warm (>23°C) is, zal het aantal schoten ... zijn dan gemiddeld.",
    "Als er een derby is, of rivalen spelen, zal het aantal schoten van beide teams ... zijn dan gemiddeld."
]

for col in causale_vragen:
    if col in enquete_df.columns:
        plt.figure()
        sns.countplot(data=enquete_df, x=col, order=["meer","zelfde","minder"], palette="Set2")
        plt.title(f"Verdeling antwoorden: {col}")
        plt.ylabel("Aantal respondenten")
        plt.show()

## 11. Experts vs Model vergelijken

- Kansvoorspellingen (Brier score)
- Doelpunten (MAE / RMSE)
- Visualisatie (scatterplots, barplots)

## 12. Confidence vs Accuracy

- Confidence bins
- Accuracy per bin
- Calibration plots
- Analyse van over- of underconfidence

## 13. Bias analyse

- Thuisvoordeel overschatting
- Motivatie overschatting
- Vorm, rangverschil
- Histogrammen / visualisaties

## 14. Visualisaties

- Factor importance
- Experts vs model resultaten
- Doelpunten vergelijking
- Confidence calibration
- Interactieve grafieken (optioneel)

## 15. Conclusie / Discussie

- Belangrijkste bevindingen
- Factoren met grootste effect
- Verschillen experts vs model
- Limitaties
- Suggesties vervolgonderzoek